# Serial echocardiographic surveillance after Kawasaki disease, classified by AHA 2017 coronary Z-score categories
## Analysis notebook v4 (Wuhan Children's Hospital KD registry, 30-day post-discharge re-export) — results lock under AHA definitions

**Why v4.** v1–v3 classified coronary status from the sonographer's words. `KD_coronary_AHA_v1.ipynb` showed that the words are shifted
by one level against the AHA 2017 Z-score categories ("扩张" is mostly a small/medium aneurysm; "正常值上限" is mostly dilation only) and
recovered the AHA category of every report from written Z-scores, Z-scores computed from written diameters with the laboratory's own
equations, numbers transcribed into notes, and — where no number exists — a calibrated wording model. v4 repeats the surveillance analyses
with those categories.

**Design (locked; change here and re-run).**
* AHA 2017 category per report (from `output_kd_coronary_aha_v1/per_report_AHA_classification.csv`): 0 no involvement (Z < 2);
  1 dilation only (2 to < 2.5); 2 small aneurysm (2.5 to < 5); 3 medium (5 to < 10); 4 large/giant (Z ≥ 10 or ≥ 8 mm). Child level =
  maximum in the window. "Involvement" = category ≥ 1; "aneurysm" = category ≥ 2.
* **Plan B (numeric era).** From the month in which fewer than 90% of reports worded dilation/aneurysm still carry their numbers (computed
  below; the laboratory moved the measurements into an unexported table), categories depend on the wording model. Primary analysis = index
  admissions 2018–2024 (2021 excluded, in-hospital reports missing Feb–Dec 2021) with echocardiograms up to the end of the numeric era
  (later follow-up censored). Sensitivity analysis = all index years 2018–2026 (2021 excluded) and all echocardiograms, using the best-
  estimate category for table-only reports, plus probability-weighted estimates for the key proportions.
* Index admission, phases (in hospital; W1 1–30 d, W2 31–90 d, W3 91–365 d, W4 > 365 d after discharge), IVIG resistance (second IVIG ≥ 36 h
  after the first, or rescue steroid/infliximab > 24 h after the first IVIG) and laboratory features as in v3.
* Uncomplicated group = in-hospital maximum category 0 (Z < 2 on every in-hospital report). Primary surveillance outcome = AHA aneurysm
  (Z ≥ 2.5) at the first post-discharge echo ≤ 30 d; secondary = any involvement (Z ≥ 2) and medium-or-larger (Z ≥ 5); sequential yields at
  W2–W4; risk factors by Firth logistic regression.
* Course of in-hospital aneurysms by size: regression below the aneurysm threshold (Z < 2.5) and to a normal lumen (Z < 2), and progression
  to a larger category, by Kaplan–Meier (time 0 = discharge; events dated at the reporting echo; censoring at the last echo).

**How to run.** Put this notebook in the folder with `川崎病重新导出30天_1/2/3.xlsx`; run `KD_coronary_AHA_v1.ipynb` first (it writes the
per-report AHA categories this notebook reads); optionally add `KD_Infant_Master_v1.3.xlsx`. Run all cells. Outputs go to
`./output_kd_echo_surveillance_v4/`. Every number is computed.

In [ ]:
import os, re, glob, json, time, warnings, shutil, hashlib
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from scipy import stats
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60); pd.set_option('display.max_colwidth', 80)

DATA_DIR = '.'
OUT_DIR = os.path.join(DATA_DIR, 'output_kd_echo_surveillance_v4')
CACHE = os.path.join(OUT_DIR, 'cache')
os.makedirs(CACHE, exist_ok=True)
# reuse the screening notebook's cache if it is present in the same folder
for alt in [os.path.join(DATA_DIR, d, 'cache') for d in ['output_kd_coronary_aha_v1', 'output_kd_echo_surveillance_v3', 'output_kd_echo_surveillance_v2', 'output_kd_echo_surveillance', 'output_kd30_screening']]:
    if os.path.isdir(alt):
        for f in glob.glob(os.path.join(alt, '*.parquet')):
            if not os.path.exists(os.path.join(CACHE, os.path.basename(f))):
                shutil.copy(f, CACHE)

plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'font.size': 11,
                     'axes.labelsize': 12, 'axes.labelweight': 'bold', 'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.facecolor': 'white', 'axes.facecolor': 'white', 'axes.grid': False, 'savefig.dpi': 300, 'pdf.fonttype': 42})
BLACK, DGREY, MGREY, LGREY = '#000000', '#4d4d4d', '#8c8c8c', '#c8c8c8'

files = sorted(glob.glob(os.path.join(DATA_DIR, '川崎病重新导出30天_*.xlsx')))
assert len(files) == 3, f'expected the three workbooks in {os.path.abspath(DATA_DIR)}, found {files}'
print('workbooks:', [os.path.basename(f) for f in files])

def wilson(k, n, z=1.96):
    """Wilson score 95% CI for a proportion; returns (lower, upper) in percent."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z**2 / n; c = p + z**2 / (2 * n); h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return (max((c - h) / d * 100, 0.0), min((c + h) / d * 100, 100.0))

def fmt_rate(k, n):
    lo, hi = wilson(k, n)
    return f'{k}/{n} ({k/n*100:.2f}%; 95% CI {lo:.2f}–{hi:.2f})' if n else 'NA'

def firth_logit(X, y, max_iter=200, tol=1e-8):
    """Firth-penalized logistic regression (Jeffreys prior). X without intercept. Returns beta, se, penalized log-likelihood."""
    X = np.column_stack([np.ones(len(X)), np.asarray(X, float)]); y = np.asarray(y, float)
    n, k = X.shape; beta = np.zeros(k)
    for _ in range(max_iter):
        eta = X @ beta; p = 1 / (1 + np.exp(-eta)); W = p * (1 - p)
        I = (X * W[:, None]).T @ X
        Iinv = np.linalg.pinv(I)
        h = np.einsum('ij,jk,ik->i', X * W[:, None], Iinv, X)
        U = X.T @ (y - p + h * (0.5 - p))
        step = Iinv @ U
        beta = beta + step
        if np.max(np.abs(step)) < tol:
            break
    eta = X @ beta; p = 1 / (1 + np.exp(-eta)); W = p * (1 - p)
    I = (X * W[:, None]).T @ X; se = np.sqrt(np.diag(np.linalg.pinv(I)))
    ll = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)) + 0.5 * np.linalg.slogdet(I)[1]
    return beta, se, ll

def firth_table(df, y, cols, labels):
    d = df[cols + [y]].dropna()
    X = d[cols].astype(float).values; yy = d[y].astype(float).values
    b, se, ll = firth_logit(X, yy)
    rows = []
    for i, (c, lab) in enumerate(zip(cols, labels), start=1):
        rows.append(dict(variable=lab, n_with=int(d[c].sum()), events_with=int(d.loc[d[c] == 1, y].sum()), events_without=int(d.loc[d[c] == 0, y].sum()),
                         OR=np.exp(b[i]), CI_low=np.exp(b[i] - 1.96 * se[i]), CI_high=np.exp(b[i] + 1.96 * se[i]), p=2 * (1 - stats.norm.cdf(abs(b[i] / se[i])))))
    return pd.DataFrame(rows), len(d), int(yy.sum())

## 1. Load sheets (cached as parquet after the first run)

In [ ]:
NEEDED = {'患者基本信息', '病案首页基本信息', '病案出院诊断编目前', '儿童心脏彩超', '住院用药医嘱', '住院病历出院记录',
          '实验室检查_血常规', '实验室检查_血C反应蛋白（CRP）测定', '实验室检查_血液肝功能测定', '实验室检查_血液电解质测定',
          '实验室检查_血红细胞沉降率测定', '实验室检查_血液N末端B型钠尿肽前体测定'}
def safe(name):
    return re.sub(r'[^\w一-鿿]+', '_', name)
def load_sheet(name):
    p = os.path.join(CACHE, safe(name) + '.parquet')
    if os.path.exists(p):
        return pd.read_parquet(p).astype('str')
    import openpyxl
    for f in files:
        wb = openpyxl.load_workbook(f, read_only=True)
        if name in wb.sheetnames:
            ws = wb[name]; rows = ws.iter_rows(values_only=True); hdr = next(rows)
            cols = [str(c) if c is not None else f'col{i}' for i, c in enumerate(hdr)]
            df = pd.DataFrame(list(rows), columns=cols).dropna(how='all')
            for c in df.columns:
                df[c] = df[c].astype('string')
            wb.close(); df.to_parquet(p, index=False)
            return df.astype('str')
        wb.close()
    raise KeyError(name)
t0 = time.time(); T = {n: load_sheet(n) for n in NEEDED}
print({n: T[n].shape for n in NEEDED}); print(f'loaded in {time.time()-t0:.0f} s')

## 2. Index admissions, treatments, laboratory features

In [ ]:
KD_PAT = r'川崎|皮肤粘膜淋巴结|皮肤黏膜淋巴结|黏膜皮肤淋巴结|粘膜皮肤淋巴结|kawasaki'
adm = T['病案首页基本信息'].copy()
adm['入院日期'] = pd.to_datetime(adm['入院日期'], errors='coerce'); adm['出院日期'] = pd.to_datetime(adm['出院日期'], errors='coerce')
adm['LOS'] = pd.to_numeric(adm['实际住院天数'], errors='coerce'); adm['age'] = pd.to_numeric(adm['年龄（岁）'], errors='coerce')
DATA_END = adm['出院日期'].max().normalize()
dx = T['病案出院诊断编目前'].copy()
dx['is_kd'] = dx['诊断疾病名称'].str.contains(KD_PAT, case=False, na=False)
adm['kd_dx'] = adm['科研就诊编号'].map(dx.groupby('科研就诊编号')['is_kd'].any()).fillna(False).astype(bool)
adm = adm.sort_values(['科研患者编号', '入院日期'])
idx = adm[adm['kd_dx']].groupby('科研患者编号').first().reset_index()[['科研患者编号', '科研就诊编号', '入院日期', '出院日期']]
idx.columns = ['科研患者编号', 'index_enc', 'idx_adm', 'idx_dis']
adm = adm.merge(idx, on='科研患者编号', how='left')
adm['days_from_idx_dis'] = (adm['入院日期'] - adm['idx_dis']).dt.total_seconds() / 86400
adm['rel'] = np.select([adm['科研就诊编号'] == adm['index_enc'], adm['index_enc'].isna(), adm['入院日期'] < adm['idx_adm'],
                        (adm['days_from_idx_dis'] >= 0) & (adm['days_from_idx_dis'] <= 30), adm['days_from_idx_dis'] > 30],
                       ['index', 'no_kd_index', 'before_index', 'within30', 'after30'], 'other')
print('inpatient encounters:', len(adm), '| patients:', adm['科研患者编号'].nunique(), '| data end:', DATA_END.date(), '|', adm['rel'].value_counts().to_dict())

pt = T['患者基本信息'].drop_duplicates('科研患者编号').copy(); pt['出生日期'] = pd.to_datetime(pt['出生日期'], errors='coerce')
m = idx.merge(adm.loc[adm['rel'] == 'index', ['科研就诊编号', 'age', 'LOS']].rename(columns={'科研就诊编号': 'index_enc'}), on='index_enc', how='left')
m = m.merge(pt[['科研患者编号', '性别', '出生日期']], on='科研患者编号', how='left')
m['male'] = (m['性别'] == '男性').astype(float)
m['age_y'] = m['age'].fillna((m['idx_adm'] - m['出生日期']).dt.days / 365.25); m['age_m'] = m['age_y'] * 12
m['year'] = m['idx_adm'].dt.year
# KD phenotype from the index discharge diagnoses
dxi = dx[dx['科研就诊编号'].isin(m['index_enc'])].groupby('科研就诊编号')['诊断疾病名称'].apply(lambda s: '|'.join(s.astype(str)))
m['dx_text'] = m['index_enc'].map(dxi).fillna('')
m['incomplete_kd'] = m['dx_text'].str.contains('不典型|不完全')
m['kd_shock'] = m['dx_text'].str.contains('休克')
# fever days at admission (admission narrative)
dr = T['住院病历出院记录'].drop_duplicates('科研就诊编号')
def fever_days(t):
    if not isinstance(t, str):
        return np.nan
    for p in [r'(?:间断|持续|反复|不规则|再次|仍)?(?:发热|发烧|高热|热)\s*(\d+(?:\.\d+)?)\s*(?:\+|余|多)?\s*天',
              r'发热\s*(\d+)\s*[-~至]\s*\d+\s*天', r'热\s*(\d+)\s*(?:\+|余|多)?\s*[dD]']:
        mm = re.search(p, t.replace('＋', '+'))
        if mm:
            v = float(mm.group(1))
            if 0 < v <= 60:
                return v
    mm = re.search(r'发热\s*(\d+)\s*小时', t)
    return round(float(mm.group(1)) / 24, 1) if mm else np.nan
dr['fever_days_adm'] = dr['入院情况'].map(fever_days)
m = m.merge(dr[['科研就诊编号', 'fever_days_adm']].rename(columns={'科研就诊编号': 'index_enc'}), on='index_enc', how='left')
m['dis_illday'] = m['fever_days_adm'] + m['LOS']

# medications in the index admission
med = T['住院用药医嘱'].copy()
med['t0'] = pd.to_datetime(med['住院用药医嘱开始时间'], errors='coerce').fillna(pd.to_datetime(med['住院用药医嘱开立时间'], errors='coerce'))
nm = med['住院用药医嘱药品名称'].fillna('')
med['is_ivig'] = nm.str.contains('静注人免疫球蛋白|静脉注射用人免疫球蛋白|人免疫球蛋白注射液|静注人血丙球', regex=True)
med['is_ster'] = nm.str.contains('甲泼尼龙|甲基强的松龙|泼尼松|强的松|地塞米松|氢化可的松', regex=True) & ~nm.str.contains('眼|滴|膏|喷', regex=True)
med['is_ifx'] = nm.str.contains('英夫利西|英夫利昔', regex=True)
med['is_anticoag'] = nm.str.contains('肝素|华法林|氯吡格雷', regex=True)
med['is_asa'] = nm.str.contains('阿司匹林|阿斯匹林', regex=True)
mi = med[med['科研就诊编号'].isin(m['index_enc'])]
def ivig_courses(g):
    ts = sorted(g['t0'].dropna().tolist()); courses = [ts[0]]
    for t in ts[1:]:
        if (t - courses[-1]).total_seconds() / 3600 >= 36:
            courses.append(t)
    return pd.Series(dict(ivig_first=courses[0], ivig_n=len(courses)))
iv = mi[mi['is_ivig']].groupby('科研就诊编号').apply(ivig_courses).reset_index().rename(columns={'科研就诊编号': 'index_enc'})
m = m.merge(iv, on='index_enc', how='left')
m['ivig_n'] = m['ivig_n'].fillna(0); m['ivig_given'] = m['ivig_n'] > 0; m['ivig_2nd'] = m['ivig_n'] >= 2
m['days_adm_to_ivig'] = (m['ivig_first'] - m['idx_adm']).dt.total_seconds() / 86400
m['fever_to_ivig'] = m['fever_days_adm'] + m['days_adm_to_ivig']
m['ivig_to_dis'] = (m['idx_dis'] - m['ivig_first']).dt.total_seconds() / 86400
for flag in ['is_ster', 'is_ifx', 'is_anticoag', 'is_asa']:
    m = m.merge(mi[mi[flag]].groupby('科研就诊编号')['t0'].min().rename(flag[3:] + '_first'), left_on='index_enc', right_index=True, how='left')
m['ster_any'] = m['ster_first'].notna()
m['ster_after_ivig'] = m['ster_first'] > m['ivig_first'] + pd.Timedelta(hours=24)
m['ifx_any'] = m['ifx_first'].notna(); m['anticoag'] = m['anticoag_first'].notna(); m['asa_any'] = m['asa_first'].notna()
m['ivig_resistant'] = m['ivig_2nd'] | (m['ster_after_ivig'] & m['ivig_given']) | m['ifx_any']

def lab_features(sheet, cols):
    lab = T[sheet].rename(columns={'患者编号': '科研患者编号'}).copy()
    lab['t'] = pd.to_datetime(lab['报告时间'], errors='coerce')
    for c in cols:
        lab[c] = pd.to_numeric(lab[c].astype(str).str.replace('[<>＜＞]', '', regex=True), errors='coerce')
    x = lab.merge(m[['科研患者编号', 'idx_adm', 'idx_dis']], on='科研患者编号', how='inner')
    x['d_adm'] = (x['t'] - x['idx_adm']).dt.total_seconds() / 86400; x['d_dis'] = (x['t'] - x['idx_dis']).dt.total_seconds() / 86400
    inidx = x[(x['d_adm'] >= -0.5) & (x['d_dis'] <= 0.5)].sort_values('t'); post = x[(x['d_dis'] > 0.5) & (x['d_dis'] <= 30)].sort_values('t')
    out = m[['科研患者编号']].copy()
    for c, s in cols.items():
        g = inidx.dropna(subset=[c]).groupby('科研患者编号')[c]; gp = post.dropna(subset=[c]).groupby('科研患者编号')
        out[f'{s}_adm'] = out['科研患者编号'].map(g.first()); out[f'{s}_dis'] = out['科研患者编号'].map(g.last()); out[f'{s}_max'] = out['科研患者编号'].map(g.max())
        out[f'{s}_p30'] = out['科研患者编号'].map(gp[c].first())
    return out.drop(columns=['科研患者编号'])
m = pd.concat([m, lab_features('实验室检查_血常规', {'白细胞计数定量-定量结果': 'WBC', '血小板计数定量-定量结果': 'PLT', '血红蛋白浓度定量-定量结果': 'Hb', '中性粒细胞总数定量-定量结果': 'NEUT'}),
               lab_features('实验室检查_血C反应蛋白（CRP）测定', {'C反应蛋白定量-定量结果': 'CRP', '超敏C反应蛋白定量-定量结果': 'hsCRP'}),
               lab_features('实验室检查_血液肝功能测定', {'谷丙转氨酶定量-定量结果': 'ALT', '白蛋白定量-定量结果': 'ALB'}),
               lab_features('实验室检查_血液电解质测定', {'钠定量-定量结果': 'Na'}),
               lab_features('实验室检查_血红细胞沉降率测定', {'血沉定量-定量结果': 'ESR'}),
               lab_features('实验室检查_血液N末端B型钠尿肽前体测定', {'N末端B型利钠肽原定量-定量结果': 'NTproBNP'})], axis=1)
for suf in ['adm', 'dis', 'max', 'p30']:
    m[f'CRPc_{suf}'] = m[f'CRP_{suf}'].fillna(m[f'hsCRP_{suf}'])
# 30-day readmission
w = adm[adm['rel'] == 'within30']
m['readm30'] = m['科研患者编号'].isin(w['科研患者编号'])
m['readm30_ivig'] = m['科研患者编号'].isin(w.loc[w['科研就诊编号'].isin(med.loc[med['is_ivig'], '科研就诊编号']), '科研患者编号'])
print('index KD admissions:', len(m), '| IVIG given', f"{m['ivig_given'].mean()*100:.1f}%", '| IVIG-resistant (proxy)', f"{m['ivig_resistant'].mean()*100:.1f}%",
      '| incomplete KD', f"{m['incomplete_kd'].mean()*100:.1f}%", '| fever days parsed', f"{m['fever_days_adm'].notna().mean()*100:.1f}%")

## 3. Echo reports → AHA category (from `KD_coronary_AHA_v1`), phases, and the numeric era

In [ ]:
AHA_FILE = os.path.join(DATA_DIR, 'output_kd_coronary_aha_v1', 'per_report_AHA_classification.csv')
assert os.path.exists(AHA_FILE), 'Run KD_coronary_AHA_v1.ipynb first: it writes output_kd_coronary_aha_v1/per_report_AHA_classification.csv'


def txt(s):
    s = s.astype(object).where(s.notna(), '').astype(str)
    return s.where(~s.isin(['None', 'nan', '<NA>', 'NaT']), '')


def report_key(pid, t, f):
    return f"{pid}|{pd.Timestamp(t).strftime('%Y-%m-%d %H:%M:%S')}|{hashlib.md5(str(f).encode('utf-8')).hexdigest()[:10]}"


e = T['儿童心脏彩超'].copy()
e['儿童超声心动图所见'] = txt(e['儿童超声心动图所见']); e['儿童超声心动图结论'] = txt(e['儿童超声心动图结论'])
e['检查时间'] = pd.to_datetime(e['检查时间'], errors='coerce'); e['EF'] = pd.to_numeric(e['射血分数'], errors='coerce')
e['report_key'] = [report_key(a, b, c) for a, b, c in zip(e['科研患者编号'], e['检查时间'], e['儿童超声心动图所见'])]
AC = pd.read_csv(AHA_FILE, dtype={'report_key': str}, usecols=['report_key', 'cor_status', 'upper_normal', 'maxZ', 'maxZ_src', 'tier', 'floor',
                                                              'P0', 'P1', 'P2', 'P3', 'P4', 'aha_best', 'p_aneurysm', 'p_large']).drop_duplicates('report_key')
e = e.merge(AC, on='report_key', how='left', validate='many_to_one')
print('echo reports:', len(e), '| matched to the AHA classification:', int(e['tier'].notna().sum()))
assert e['tier'].notna().all(), 'report keys do not match the AHA classification output'
e['w'] = e['cor_status'].map({'normal': 0, 'dilation': 1, 'aneurysm': 2})          # sonographer wording (v3 reading)
e['upper_normal'] = e['upper_normal'].astype(str).str.lower().eq('true')
e['cat'] = e['aha_best']                                                              # AHA category 0-4 (NaN = unclassifiable)
e['p_inv'] = 1 - e['P0']                                                              # P(category >= 1)
e['p_med'] = e['P3'] + e['P4']                                                        # P(category >= 3)
e = e.merge(m[['科研患者编号', 'index_enc', 'idx_adm', 'idx_dis', 'fever_days_adm']], on='科研患者编号', how='left')
e['d_adm'] = (e['检查时间'] - e['idx_adm'].dt.normalize()).dt.days; e['d_dis'] = (e['检查时间'] - e['idx_dis'].dt.normalize()).dt.days
e['ill_day'] = e['fever_days_adm'] + e['d_adm']
e['phase'] = np.select([e['index_enc'].isna(), e['d_adm'] < 0, (e['d_adm'] >= 0) & (e['d_dis'] <= 0), (e['d_dis'] > 0) & (e['d_dis'] <= 30),
                        (e['d_dis'] > 30) & (e['d_dis'] <= 90), (e['d_dis'] > 90) & (e['d_dis'] <= 365), e['d_dis'] > 365],
                       ['no_kd_index', 'pre_index', 'in_index', 'post30', 'd31_90', 'd91_365', 'gt365'], 'other')
e = e.sort_values(['科研患者编号', '检查时间']).reset_index(drop=True)
CATS = ['None', 'Dilation only', 'Small', 'Medium', 'Large/giant']
CATL = ['No involvement (Z<2)', 'Dilation only (2–<2.5)', 'Small aneurysm (2.5–<5)', 'Medium aneurysm (5–<10)', 'Large/giant (≥10 or ≥8 mm)']
print('reports by AHA category:', e['cat'].value_counts().sort_index().to_dict(), '| evidence tier:', e['tier'].value_counts().sort_index().to_dict())

# numeric era: the first month (from 2024) in which <90% of reports worded dilation/aneurysm still carry their numbers (tiers 1-3)
ab = e[(e['w'] >= 1) & (e['检查时间'] >= '2024-01-01')].copy(); ab['ym'] = ab['检查时间'].dt.to_period('M')
mon = ab.groupby('ym').agg(n=('tier', 'size'), exact_pct=('tier', lambda s: s.isin([1, 2, 3]).mean() * 100))
first_table_month = mon.index[mon['exact_pct'] < 90][0]
ERA_END = first_table_month.to_timestamp() - pd.Timedelta(days=1)
mon.round(1).to_csv(os.path.join(OUT_DIR, 'TableS0_numeric_coverage_by_month.csv'))
print(mon.round(1).T.to_string()); print('numeric era ends:', ERA_END.date(), '(first month below 90%:', str(first_table_month) + ')')

## 4. Per-child window features and the two analysis scopes (primary = numeric era; sensitivity = all data)

In [ ]:
def window_feats(sub, name):
    sub = sub.sort_values('检查时间'); g = sub.groupby('科研患者编号')
    return pd.DataFrame({f'{name}_n': g.size(), f'{name}_cat_max': g['cat'].max(), f'{name}_cat_first': g['cat'].first(), f'{name}_cat_last': g['cat'].last(),
                         f'{name}_z_max': g['maxZ'].max(), f'{name}_z_first': g['maxZ'].first(), f'{name}_day_first': g['d_dis'].first(), f'{name}_day_last': g['d_dis'].last(),
                         f'{name}_illday_first': g['ill_day'].first(), f'{name}_illday_last': g['ill_day'].last(), f'{name}_w_max': g['w'].max(),
                         f'{name}_upper': g['upper_normal'].max(), f'{name}_EF_min': g['EF'].min(), f'{name}_tier4_any': g['tier'].apply(lambda s: bool((s == 4).any())),
                         f'{name}_pan_first': g['p_aneurysm'].first(), f'{name}_pinv_first': g['p_inv'].first(), f'{name}_pmed_first': g['p_med'].first(),
                         f'{name}_pan_max': g['p_aneurysm'].max(), f'{name}_pinv_max': g['p_inv'].max()})


def build_scope(scope):
    """returns (child frame, report frame) for 'primary' (index 2018-2024 excl. 2021; echoes up to ERA_END) or 'all' (2018-2026 excl. 2021; all echoes)"""
    if scope == 'primary':
        mm_ = m[(m['year'] >= 2018) & (m['year'] <= 2024) & (m['year'] != 2021)].copy(); ee = e[e['cat'].notna() & (e['检查时间'] <= ERA_END)]
    else:
        mm_ = m[(m['year'] >= 2018) & (m['year'] != 2021)].copy(); ee = e[e['cat'].notna()]
    ee = ee[ee['科研患者编号'].isin(mm_['科研患者编号'])].copy()
    for ph in ['in_index', 'post30', 'd31_90', 'd91_365', 'gt365']:
        mm_ = mm_.merge(window_feats(ee[ee['phase'] == ph], ph), left_on='科研患者编号', right_index=True, how='left')
    mm_ = mm_.merge(window_feats(ee[(ee['d_dis'] > 0) & (ee['d_dis'] <= 60)], 'post60'), left_on='科研患者编号', right_index=True, how='left')
    mm_ = mm_.merge(window_feats(ee[(ee['d_dis'] > 30) & (ee['d_dis'] <= 365)], 'd31_365'), left_on='科研患者编号', right_index=True, how='left')
    mm_ = mm_.merge(window_feats(ee[(ee['d_dis'] > 0) & (ee['ill_day'] >= 11) & (ee['ill_day'] <= 21)], 'w2'), left_on='科研患者编号', right_index=True, how='left')
    mm_ = mm_.merge(window_feats(ee[(ee['d_dis'] > 0) & (ee['ill_day'] >= 22) & (ee['ill_day'] <= 56)], 'w6'), left_on='科研患者编号', right_index=True, how='left')
    for c in [c for c in mm_.columns if c.endswith('_n')]:
        mm_[c] = mm_[c].fillna(0)
    mm_['age_lt12m'] = mm_['age_m'] < 12; mm_['age_lt6m'] = mm_['age_m'] < 6
    mm_['crp_dis_gt20'] = mm_['CRPc_dis'] > 20; mm_['wbc_dis_gt12'] = mm_['WBC_dis'] > 12
    mm_['inflam_dis'] = (mm_['crp_dis_gt20'] | mm_['wbc_dis_gt12']).fillna(False)
    mm_['echo_ge2'] = mm_['in_index_n'] >= 2
    mm_['in_max'] = mm_['in_index_cat_max']
    return mm_, ee


SC = {s: build_scope(s) for s in ['primary', 'all']}
for s, (mm_, ee) in SC.items():
    print(s, '| index admissions', len(mm_), '| with in-hospital classifiable echo', int((mm_['in_index_n'] > 0).sum()), '| reports', len(ee),
          '| reports from the wording model among dilation/aneurysm wording:', f"{((ee['w'] >= 1) & (ee['tier'] == 4)).sum()}/{(ee['w'] >= 1).sum()}")

## 5. Core analyses (run for both scopes)
R0 cohort and in-hospital category; R1 where aneurysms are first documented; R2 sequential yields after an uncomplicated (Z < 2) in-hospital
course; R3 in-hospital dilation only; R4 course of in-hospital aneurysms by size; R5 status by illness-day window. Probability-weighted
versions of the key proportions use the category probabilities of each report (exact tiers are 0/1).

In [ ]:
WORDER = ['index admission (or before)', 'W1 1–30 d', 'W2 31–90 d', 'W3 91–365 d', 'W4 >365 d']


def rate(k, n):
    lo, hi = wilson(k, n)
    return dict(events=int(k), n=int(n), pct=round(k / n * 100, 2) if n else np.nan, ci_low=round(lo, 2), ci_high=round(hi, 2))


def analyze(scope):
    mm_, ee = SC[scope]
    R = {'scope': scope}
    c1 = mm_[mm_['in_index_n'] > 0].copy()
    R['C1_n'] = len(c1)
    R['inhosp_cat'] = {CATS[int(k)]: int(v) for k, v in c1['in_max'].value_counts().sort_index().items()}
    R['inhosp_wording_x_cat'] = pd.crosstab(c1['in_index_w_max'].map({0: 'normal', 1: 'dilation', 2: 'aneurysm'}), c1['in_max'].map(dict(enumerate(CATS))))
    R['inhosp_aneurysm'] = rate(int((c1['in_max'] >= 2).sum()), len(c1))
    ec = ee[ee['科研患者编号'].isin(c1['科研患者编号'])].copy()
    ec['window'] = np.select([ec['phase'].isin(['pre_index', 'in_index']), ec['phase'] == 'post30', ec['phase'] == 'd31_90', ec['phase'] == 'd91_365', ec['phase'] == 'gt365'], WORDER, 'other')
    ec = ec[ec['window'] != 'other']
    inmax = c1.set_index('科研患者编号')['in_max']
    # ---- R1 first documentation ----
    first = {}
    for th in (1, 2, 3):
        f = ec[ec['cat'] >= th].sort_values('检查时间').groupby('科研患者编号').first(); f['in_max'] = f.index.map(inmax); first[th] = f
    R['first_doc'] = pd.DataFrame({f'cat>={th}': first[th]['window'].value_counts().reindex(WORDER).fillna(0).astype(int) for th in (1, 2, 3)})
    fa = first[2]; pa = fa[(fa['in_max'] < 2) & (fa['window'] != WORDER[0])]   # first documented in a post-discharge window
    R['aneurysm_before_index_only'] = int(((fa['in_max'] < 2) & (fa['window'] == WORDER[0])).sum())   # pre-index aneurysm, not on index-admission reports
    R['aneurysm_ever'] = int(len(fa)); R['aneurysm_first_after_discharge'] = int(len(pa))
    R['aneurysm_first_after_discharge_by_inhosp'] = {CATS[int(k)]: int(v) for k, v in pa['in_max'].value_counts().sort_index().items()}
    R['aneurysm_first_after_discharge_by_window'] = pa['window'].value_counts().reindex(WORDER[1:]).fillna(0).astype(int).to_dict()
    R['pa_table'] = pa
    followed = c1[c1[['post30_n', 'd31_90_n', 'd91_365_n', 'gt365_n']].sum(axis=1) > 0]
    R['followed_n'] = len(followed)
    rows = []
    for k in (0, 1):
        n_f = int((followed['in_max'] == k).sum()); kk = int((pa['in_max'] == k).sum()); lo, hi = wilson(kk, n_f)
        rows.append(dict(in_hospital=CATS[k], followed=n_f, post_discharge_first_aneurysm=kk, per_1000=round(kk / n_f * 1000, 1) if n_f else np.nan, ci_low=round(lo * 10, 1), ci_high=round(hi * 10, 1)))
    R['pd_first_rates'] = pd.DataFrame(rows)
    post_max = ec[ec['d_dis'] > 0].groupby('科研患者编号')['cat'].max()
    fw = followed.assign(post_max=followed['科研患者编号'].map(post_max))
    R['worsening'] = pd.DataFrame([dict(in_hospital=CATS[k], followed=int((fw['in_max'] == k).sum()), higher_category_after_discharge=int(((fw['in_max'] == k) & (fw['post_max'] > k)).sum()),
                                        new_aneurysm=int(((fw['in_max'] == k) & (fw['post_max'] >= 2)).sum()) if k < 2 else np.nan) for k in range(5)])
    # ---- R2 sequential yields after an uncomplicated in-hospital course (category 0) ----
    base = c1[c1['in_max'] == 0].copy(); coh = base[base['post30_n'] > 0].copy()
    R['uncomplicated_n'] = len(base); R['surv_n'] = len(coh); R['no_echo_30d'] = int(len(base) - len(coh))
    coh['new_inv'] = coh['post30_cat_first'] >= 1; coh['new_an'] = coh['post30_cat_first'] >= 2; coh['new_med'] = coh['post30_cat_first'] >= 3
    coh['any_inv30'] = coh['post30_cat_max'] >= 1; coh['any_an30'] = coh['post30_cat_max'] >= 2
    seq = []
    def add(step, k, n, extra=None):
        d = dict(step=step); d.update(rate(k, n)); d.update(extra or {}); seq.append(d)
    add('First echo ≤30 d after an uncomplicated in-hospital course: aneurysm (Z≥2.5)', int(coh['new_an'].sum()), len(coh), dict(prob_weighted_events=round(float(coh['post30_pan_first'].sum()), 1)))
    add('   … any involvement (Z≥2)', int(coh['new_inv'].sum()), len(coh), dict(prob_weighted_events=round(float(coh['post30_pinv_first'].sum()), 1)))
    add('   … medium or larger (Z≥5)', int(coh['new_med'].sum()), len(coh), dict(prob_weighted_events=round(float(coh['post30_pmed_first'].sum()), 1)))
    add('   … aneurysm on any echo ≤30 d', int(coh['any_an30'].sum()), len(coh))
    s2n = coh[(coh['post30_cat_max'] == 0) & (coh['d31_90_n'] > 0)]
    add('31–90 d after uncomplicated in-hospital + normal ≤30-d echoes: any involvement', int((s2n['d31_90_cat_max'] >= 1).sum()), len(s2n))
    add('   … aneurysm', int((s2n['d31_90_cat_max'] >= 2).sum()), len(s2n))
    s3n = coh[(coh['post30_cat_max'] == 0) & (coh['d31_90_cat_max'] == 0) & (coh['d91_365_n'] > 0)]
    add('91–365 d after normal findings at all three prior windows: any involvement', int((s3n['d91_365_cat_max'] >= 1).sum()), len(s3n))
    add('   … aneurysm', int((s3n['d91_365_cat_max'] >= 2).sum()), len(s3n))
    s4n = coh[(coh['post30_cat_max'] == 0) & (coh['d31_90_cat_max'] == 0) & (coh['d91_365_cat_max'] == 0) & (coh['gt365_n'] > 0)]
    add('>365 d after normal findings at all four prior windows: any involvement', int((s4n['gt365_cat_max'] >= 1).sum()), len(s4n))
    add('   … aneurysm', int((s4n['gt365_cat_max'] >= 2).sum()), len(s4n))
    b2 = base[base['d31_90_n'] > 0]
    add('31–90 d after an uncomplicated in-hospital course (with or without a ≤30-d echo): aneurysm', int((b2['d31_90_cat_max'] >= 2).sum()), len(b2))
    R['seq'] = pd.DataFrame(seq); R['coh'] = coh; R['base'] = base; R['c1'] = c1; R['ec'] = ec; R['first'] = first; R['post_max'] = post_max
    # ---- R3 in-hospital dilation only (Z 2-<2.5) and the 'Z <2.5 in hospital' population ----
    dz = c1[(c1['in_max'] == 1) & (c1['post30_n'] > 0)]
    R['dilation_only_first_echo_aneurysm'] = rate(int((dz['post30_cat_first'] >= 2).sum()), len(dz))
    R['dilation_only_first_echo_normal'] = rate(int((dz['post30_cat_first'] == 0).sum()), len(dz))
    # ---- R4 course of in-hospital aneurysms by size ----
    ev_rows = []
    for k in (2, 3, 4):
        ids = c1.loc[c1['in_max'] == k, '科研患者编号']
        for lo_, hi_, wl in [(0, 30, 'W1 1–30 d'), (30, 90, 'W2 31–90 d'), (90, 365, 'W3 91–365 d'), (365, 100000, 'W4 >365 d')]:
            s = ec[(ec['d_dis'] > lo_) & (ec['d_dis'] <= hi_) & ec['科研患者编号'].isin(ids)].groupby('科研患者编号')['cat'].max()
            ev_rows.append(dict(in_hospital=CATS[k], n_in_hospital=len(ids), window=wl, with_echo=len(s), normal_lumen=int((s == 0).sum()), below_aneurysm=int((s <= 1).sum()),
                                still_aneurysm=int((s >= 2).sum()), larger=int((s > k).sum()), below_aneurysm_pct=round((s <= 1).mean() * 100, 1) if len(s) else np.nan,
                                normal_lumen_pct=round((s == 0).mean() * 100, 1) if len(s) else np.nan))
    R['evolution'] = pd.DataFrame(ev_rows)
    # ---- R5 status by illness-day window (descriptive) ----
    ecd = ec[ec['ill_day'].notna()].copy()
    DB = [0, 7, 10, 14, 21, 28, 42, 60, 90, 180, 365, 100000]; DL = ['1–7', '8–10', '11–14', '15–21', '22–28', '29–42', '43–60', '61–90', '91–180', '181–365', '>365']
    ecd['win'] = pd.cut(ecd['ill_day'], DB, labels=DL)
    occ = ecd.groupby(['科研患者编号', 'win'], observed=True)['cat'].max().reset_index()
    R['illday'] = occ.groupby('win', observed=True)['cat'].agg(children='size', involvement_pct=lambda s: round((s >= 1).mean() * 100, 2), aneurysm_pct=lambda s: round((s >= 2).mean() * 100, 2),
                                                              medium_plus_pct=lambda s: round((s >= 3).mean() * 100, 2))
    occ0 = occ[occ['科研患者编号'].isin(base['科研患者编号'])]
    R['illday_uncomplicated'] = occ0.groupby('win', observed=True)['cat'].agg(children='size', involvement_pct=lambda s: round((s >= 1).mean() * 100, 2), aneurysm_pct=lambda s: round((s >= 2).mean() * 100, 2))
    return R


RES = {s: analyze(s) for s in ['primary', 'all']}
P_, S_ = RES['primary'], RES['all']
for s, R in RES.items():
    print(f"=== {s}: C1 {R['C1_n']:,} | in-hospital {R['inhosp_cat']} | uncomplicated {R['uncomplicated_n']:,}, with ≤30-d echo {R['surv_n']:,} | aneurysm ever {R['aneurysm_ever']} (first after discharge {R['aneurysm_first_after_discharge']})")
    print(R['seq'].to_string(index=False))
print(P_['inhosp_wording_x_cat'])

## 6. Primary scope: cohort description, selection, who has an aneurysm at the first post-discharge echo, sensitivity analyses
Risk population (R3) = children discharged without an aneurysm (in-hospital maximum Z < 2.5: no involvement or dilation only) with a first
post-discharge echo ≤ 30 d; outcome = aneurysm (Z ≥ 2.5) on that echo. In-hospital dilation only is the AHA counterpart of the v3
"upper-limit-of-normal wording". Secondary model: uncomplicated group (Z < 2), outcome any involvement.

In [ ]:
def table_compare(df, group, rows_num, rows_bin):
    out = []
    g1, g0 = df[df[group]], df[~df[group]]
    for v, lab in rows_num:
        a, b = g1[v].dropna(), g0[v].dropna()
        p = stats.mannwhitneyu(a, b).pvalue if len(a) > 0 and len(b) > 0 else np.nan
        out.append(dict(variable=lab, group1=f'{a.median():.1f} ({a.quantile(.25):.1f}–{a.quantile(.75):.1f})', group0=f'{b.median():.1f} ({b.quantile(.25):.1f}–{b.quantile(.75):.1f})', n1=len(a), n0=len(b), p=p))
    for v, lab in rows_bin:
        a, b = g1[v].astype(float), g0[v].astype(float)
        ct = np.array([[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]])
        p = stats.fisher_exact(ct)[1] if ct.min() < 5 else stats.chi2_contingency(ct)[1]
        out.append(dict(variable=lab, group1=f'{int(a.sum())} ({a.mean()*100:.1f}%)', group0=f'{int(b.sum())} ({b.mean()*100:.1f}%)', n1=len(a), n0=len(b), p=p))
    t = pd.DataFrame(out); t['p'] = t['p'].map(lambda p: '<0.001' if p < 0.001 else f'{p:.3f}')
    return t


coh, base, c1 = P_['coh'], P_['base'], P_['c1']
NUM = [('age_m', 'Age, months'), ('fever_days_adm', 'Fever days at admission'), ('fever_to_ivig', 'Illness day of IVIG'), ('LOS', 'Length of stay, days'),
       ('WBC_adm', 'WBC at admission, ×10⁹/L'), ('Hb_adm', 'Hemoglobin at admission, g/L'), ('PLT_adm', 'Platelets at admission, ×10⁹/L'), ('CRPc_adm', 'CRP at admission, mg/L'),
       ('ALB_adm', 'Albumin at admission, g/L'), ('Na_adm', 'Sodium at admission, mmol/L'), ('ALT_adm', 'ALT at admission, U/L'), ('ESR_adm', 'ESR at admission, mm/h'),
       ('WBC_dis', 'WBC at discharge, ×10⁹/L'), ('CRPc_dis', 'CRP at discharge, mg/L'), ('PLT_dis', 'Platelets at discharge, ×10⁹/L'), ('PLT_max', 'Peak platelets in hospital, ×10⁹/L'),
       ('in_index_n', 'In-hospital echoes, n'), ('in_index_illday_last', 'Illness day of last in-hospital echo')]
BIN = [('male', 'Male'), ('incomplete_kd', 'Incomplete KD (diagnosis text)'), ('ivig_given', 'IVIG given'), ('ivig_2nd', 'Second IVIG course'), ('ivig_resistant', 'IVIG resistance (composite)'),
       ('ster_any', 'Any systemic corticosteroid'), ('readm30', 'Readmitted within 30 days')]
NUM1 = NUM + [('post30_day_first', 'First post-discharge echo, days after discharge'), ('post30_illday_first', 'First post-discharge echo, illness day'), ('PLT_p30', 'Platelets at first post-discharge visit, ×10⁹/L'), ('CRPc_p30', 'CRP at first post-discharge visit, mg/L')]
BIN1 = BIN + [('age_lt12m', 'Age <12 months'), ('crp_dis_gt20', 'Discharge CRP >20 mg/L'), ('wbc_dis_gt12', 'Discharge WBC >12 ×10⁹/L'), ('echo_ge2', '≥2 in-hospital echoes')]
desc = dict(n=len(coh), age_m_median=coh['age_m'].median(), age_lt12m_pct=coh['age_lt12m'].mean() * 100, male_pct=coh['male'].mean() * 100, ivig_given_pct=coh['ivig_given'].mean() * 100,
            ivig_resistant_pct=coh['ivig_resistant'].mean() * 100, year_min=int(coh['year'].min()), year_max=int(coh['year'].max()),
            post30_day_median=float(coh['post30_day_first'].median()), post30_day_IQR=[float(coh['post30_day_first'].quantile(.25)), float(coh['post30_day_first'].quantile(.75))],
            post30_illday_median=float(coh['post30_illday_first'].median()), post30_illday_IQR=[float(coh['post30_illday_first'].quantile(.25)), float(coh['post30_illday_first'].quantile(.75))])
print({k: (round(v, 1) if isinstance(v, float) else v) for k, v in desc.items()})


def yield_by(df, var, out='new_an', labels=None):
    rows = []
    for val, g in df.groupby(var, dropna=False):
        d = dict(stratum=labels.get(val, f'{var} = {val}') if labels else f'{var} = {val}'); d.update(rate(int(g[out].sum()), len(g))); rows.append(d)
    return pd.DataFrame(rows)


# R3 population: children discharged without an aneurysm (in-hospital maximum Z <2.5: none or dilation only) with a first echo <=30 d
zb = c1[(c1['in_max'] <= 1) & (c1['post30_n'] > 0)].copy()
zb['dil_only'] = zb['in_max'] == 1; zb['new_an'] = zb['post30_cat_first'] >= 2; zb['new_med'] = zb['post30_cat_first'] >= 3
zb['era'] = np.where(zb['year'] <= 2020, '2018–2020', '2022–2024')
zb['agegrp'] = pd.cut(zb['age_m'], [0, 6, 12, 24, 60, 240], labels=['<6 mo', '6–11 mo', '12–23 mo', '2–4 y', '≥5 y'])
zb['risk_count'] = zb[['age_lt12m', 'ivig_resistant', 'dil_only']].astype(int).sum(axis=1) + zb['inflam_dis'].astype(int)
zb['risk_c'] = zb['risk_count'].clip(upper=2).map({0: '0 factors', 1: '1 factor', 2: '≥2 factors'})
base_nb = c1[c1['in_max'] <= 1].copy(); base_nb['has_p30'] = base_nb['post30_n'] > 0
tS1 = table_compare(base_nb, 'has_p30', NUM, BIN).rename(columns={'group1': 'echo within 30 d', 'group0': 'no echo within 30 d'})
tS1.to_csv(os.path.join(OUT_DIR, 'TableS1_selection_echo_within30d.csv'), index=False)
t1 = table_compare(zb, 'new_an', NUM1, BIN1 + [('dil_only', 'In-hospital dilation only (Z 2–<2.5)')]).rename(columns={'group1': 'Aneurysm at first post-discharge echo', 'group0': 'No aneurysm'})
t1.to_csv(os.path.join(OUT_DIR, 'Table1_cohort_by_outcome.csv'), index=False); print(t1.to_string())
SPEC = [('dil_only', {True: 'In-hospital dilation only (Z 2–<2.5)', False: 'In-hospital no involvement (Z<2)'}),
        ('agegrp', {k: k for k in ['<6 mo', '6–11 mo', '12–23 mo', '2–4 y', '≥5 y']}), ('male', {1.0: 'Male', 0.0: 'Female'}), ('ivig_resistant', {True: 'IVIG-resistant', False: 'IVIG-responsive'}),
        ('crp_dis_gt20', {True: 'Discharge CRP >20 mg/L', False: 'Discharge CRP ≤20 mg/L'}), ('wbc_dis_gt12', {True: 'Discharge WBC >12', False: 'Discharge WBC ≤12'}),
        ('echo_ge2', {True: '≥2 in-hospital echoes', False: '1 in-hospital echo'}), ('incomplete_kd', {True: 'Incomplete KD', False: 'Complete KD'}),
        ('readm30', {True: 'Readmitted ≤30 d', False: 'Not readmitted'}), ('era', {'2018–2020': 'Index admission 2018–2020', '2022–2024': 'Index admission 2022–2024'}),
        ('risk_c', {'0 factors': '0 factors', '1 factor': '1 factor', '≥2 factors': '≥2 factors'})]
strata = pd.concat([yield_by(zb, v, 'new_an', lab) for v, lab in SPEC], ignore_index=True)
strata.to_csv(os.path.join(OUT_DIR, 'Table4a_stratified_yield.csv'), index=False); print(strata.to_string(index=False))
cols = ['dil_only', 'age_lt12m', 'male', 'ivig_resistant', 'crp_dis_gt20', 'wbc_dis_gt12', 'echo_ge2']
labels = ['In-hospital dilation only (Z 2–<2.5)', 'Age <12 months', 'Male sex', 'IVIG resistance', 'Discharge CRP >20 mg/L', 'Discharge WBC >12 ×10⁹/L', '≥2 in-hospital echoes']
fir = []
for c, lab in zip(cols, labels):
    t_, n_, ev_ = firth_table(zb, 'new_an', [c], [lab]); t_['model'] = 'univariable'; t_['population'] = 'Z<2.5 in hospital'; t_['outcome'] = 'aneurysm (Z≥2.5)'; t_['n'] = n_; t_['events'] = ev_; fir.append(t_)
MB, nB, evB = firth_table(zb, 'new_an', cols, labels)
MB['model'] = 'multivariable'; MB['population'] = 'Z<2.5 in hospital'; MB['outcome'] = 'aneurysm (Z≥2.5)'; MB['n'] = nB; MB['events'] = evB; fir.append(MB.copy())
# secondary: uncomplicated group (Z<2), outcome any involvement (Z>=2)
t_, nA, evA = firth_table(coh, 'new_inv', cols[1:], labels[1:]); t_['model'] = 'multivariable'; t_['population'] = 'Z<2 in hospital'; t_['outcome'] = 'involvement (Z≥2)'; t_['n'] = nA; t_['events'] = evA; fir.append(t_)
FIR = pd.concat(fir, ignore_index=True)
for c in ['OR', 'CI_low', 'CI_high']:
    FIR[c] = FIR[c].round(2)
FIR['p_fmt'] = FIR['p'].map(lambda p: '<0.001' if p < 0.001 else f'{p:.3f}')
FIR.to_csv(os.path.join(OUT_DIR, 'Table4b_firth_logistic.csv'), index=False); print(FIR[FIR['model'] != 'univariable'].drop(columns=['p']).to_string(index=False))
DZ = dict(none=rate(int(zb.loc[~zb['dil_only'], 'new_an'].sum()), int((~zb['dil_only']).sum())), dilation_only=rate(int(zb.loc[zb['dil_only'], 'new_an'].sum()), int(zb['dil_only'].sum())))
print('aneurysm at the first ≤30-d echo by in-hospital category:', DZ)

# sensitivity analyses (primary scope)
S = []
def sens(label, df_base, cf, cm, cn, pf=None):
    d = df_base[df_base[cn] > 0]
    r_ = dict(analysis=label); r_.update(rate(int((d[cf] >= 2).sum()), len(d))); r_['involvement'] = int((d[cf] >= 1).sum())
    r_['prob_weighted_aneurysm'] = round(float(d[pf].sum()), 1) if pf else np.nan
    S.append(r_)
sens('Primary: uncomplicated in hospital (Z<2); first echo ≤30 d', base, 'post30_cat_first', 'post30_cat_max', 'post30_n', 'post30_pan_first')
sens('S1: baseline = last in-hospital echo Z<2 (earlier involvement allowed)', c1[c1['in_index_cat_last'] == 0], 'post30_cat_first', 'post30_cat_max', 'post30_n', 'post30_pan_first')
sens('S2: first post-discharge echo ≤60 d', base, 'post60_cat_first', 'post60_cat_max', 'post60_n', 'post60_pan_first')
sens('S3: IVIG-treated only', base[base['ivig_given']], 'post30_cat_first', 'post30_cat_max', 'post30_n', 'post30_pan_first')
sens('S4: excluding readmitted ≤30 d', base[~base['readm30']], 'post30_cat_first', 'post30_cat_max', 'post30_n', 'post30_pan_first')
sens('S6a: index admissions 2018–2020', base[base['year'] <= 2020], 'post30_cat_first', 'post30_cat_max', 'post30_n', 'post30_pan_first')
sens('S6b: index admissions 2022–2024', base[base['year'] >= 2022], 'post30_cat_first', 'post30_cat_max', 'post30_n', 'post30_pan_first')
sens('S7a: first post-discharge echo on illness day 11–21', base, 'w2_cat_first', 'w2_cat_max', 'w2_n', 'w2_pan_first')
sens('S7b: illness day 22–56 after uncomplicated in-hospital + normal day-11–21 echo', base[base['w2_cat_max'] == 0], 'w6_cat_first', 'w6_cat_max', 'w6_n', 'w6_pan_first')
r_ = dict(analysis='S5: events counted only when the first ≤30-d report carries exact numbers (tiers 1–3)'); r_.update(rate(int(((coh['post30_cat_first'] >= 2) & ~coh['post30_tier4_any'].astype(bool)).sum()), len(coh))); S.append(r_)
SA, SAb = RES['all']['base'], RES['all']['coh']
r_ = dict(analysis='Plan B sensitivity: all index years 2018–2026 (2021 excluded), all echoes, wording-model categories for table-only reports'); r_.update(rate(int((SAb['post30_cat_first'] >= 2).sum()), len(SAb)))
r_['involvement'] = int((SAb['post30_cat_first'] >= 1).sum()); r_['prob_weighted_aneurysm'] = round(float(SAb['post30_pan_first'].sum()), 1); S.append(r_)
tS2 = pd.DataFrame(S); tS2.to_csv(os.path.join(OUT_DIR, 'TableS2_sensitivity.csv'), index=False); print(tS2.to_string(index=False))

## 7. Time-to-event analyses
Time 0 = discharge from the index admission; events dated at the echocardiogram that first shows them; censoring at the last echocardiogram
in the scope (primary scope: at the end of the numeric era at the latest). Cumulative incidence = 1 − Kaplan–Meier.
* TTE-1: uncomplicated in hospital (Z < 2) → first post-discharge aneurysm (Z ≥ 2.5), overall and by baseline risk-factor count; and in
  children with Z < 2.5 in hospital, dilation only vs none.
* TTE-2: in-hospital aneurysm by size (small / medium / large-giant) → first post-discharge report below the aneurysm threshold (Z < 2.5)
  and first report with a normal lumen (Z < 2).
* TTE-3: progression → first post-discharge report in a higher category (in-hospital dilation only → aneurysm; small/medium → larger), by
  IVIG resistance.
* TTE-4: illness day on which involvement / an aneurysm was first documented (whole cohort C1).

In [ ]:
def km(time, event, tgrid=None):
    """Kaplan–Meier estimate with Greenwood variance. Returns DataFrame(t, n_risk, n_event, S, se) at event times (plus t=0)."""
    t = np.asarray(time, float); d = np.asarray(event, int)
    order = np.argsort(t); t = t[order]; d = d[order]
    uniq = np.unique(t[d == 1])
    rows = [dict(t=0.0, n_risk=len(t), n_event=0, S=1.0, se=0.0)]
    S = 1.0; gw = 0.0
    for u in uniq:
        n = int((t >= u).sum()); e = int(((t == u) & (d == 1)).sum())
        if n == 0:
            continue
        S *= (1 - e / n); gw += e / (n * (n - e)) if n > e else 0.0
        rows.append(dict(t=u, n_risk=n, n_event=e, S=S, se=S * np.sqrt(gw)))
    if len(t) and t.max() > rows[-1]['t']:   # extend the curve to the last censoring time
        rows.append(dict(t=float(t.max()), n_risk=int((t >= t.max()).sum()), n_event=0, S=S, se=rows[-1]['se']))
    return pd.DataFrame(rows)

def km_at(kmdf, tq):
    """S and 95% CI (log-log) at time tq."""
    r = kmdf[kmdf['t'] <= tq].iloc[-1]; S, se = r['S'], r['se']
    if S <= 0 or S >= 1 or se == 0:
        return S, S, S
    z = 1.96; lg = np.log(-np.log(S)); v = se / (S * abs(np.log(S)))
    return S, np.exp(-np.exp(lg + z * v)), np.exp(-np.exp(lg - z * v))

def n_at_risk(time, tq):
    return int((np.asarray(time, float) >= tq).sum())

def logrank(time, event, group):
    """Log-rank test for k groups. Returns (chi2, df, p)."""
    t = np.asarray(time, float); d = np.asarray(event, int); g = np.asarray(group)
    groups = np.unique(g); k = len(groups)
    O = np.zeros(k); Ecount = np.zeros(k); V = np.zeros((k, k))
    for u in np.unique(t[d == 1]):
        at = t >= u; n = at.sum(); e = ((t == u) & (d == 1)).sum()
        ni = np.array([(at & (g == gg)).sum() for gg in groups]); ei = np.array([((t == u) & (d == 1) & (g == gg)).sum() for gg in groups])
        O += ei; Ecount += e * ni / n
        if n > 1:
            for i in range(k):
                for j in range(k):
                    V[i, j] += e * (n - e) / (n - 1) * (ni[i] / n) * ((1 if i == j else 0) - ni[j] / n)
    OE = (O - Ecount)[:-1]; Vm = V[:-1, :-1]
    chi2 = float(OE @ np.linalg.pinv(Vm) @ OE); df = k - 1
    return chi2, df, float(1 - stats.chi2.cdf(chi2, df))

def median_time(kmdf):
    below = kmdf[kmdf['S'] <= 0.5]
    return float(below['t'].iloc[0]) if len(below) else np.nan

def plot_km(ax, kmdf, label, color, ls='-', cumulative=False, ci=True):
    t = kmdf['t'].values; S = kmdf['S'].values
    y = 1 - S if cumulative else S
    ax.step(t, y * 100, where='post', color=color, lw=1.6, ls=ls, label=label)
    if ci and len(kmdf) > 1:
        lo = []; hi = []
        for _, r in kmdf.iterrows():
            s, l, h = km_at(kmdf, r['t']); lo.append(l); hi.append(h)
        lo = np.array(lo); hi = np.array(hi)
        ylo, yhi = (1 - hi, 1 - lo) if cumulative else (lo, hi)
        ax.fill_between(t, ylo * 100, yhi * 100, step='post', color=color, alpha=0.12, linewidth=0)

def risk_table(ax, groups_times, labels, tqs, y0=-0.30, dy=0.085, fs=8.5):
    """Write numbers at risk below an axis (axes coordinates)."""
    ax.text(-0.06, y0 + dy, 'No. at risk', transform=ax.transAxes, fontsize=fs, ha='right', va='center', fontweight='bold')
    for i, (times, lab) in enumerate(zip(groups_times, labels)):
        ax.text(-0.06, y0 - i * dy, lab, transform=ax.transAxes, fontsize=fs, ha='right', va='center')
        xmin, xmax = ax.get_xlim()
        for tq in tqs:
            ax.text((tq - xmin) / (xmax - xmin), y0 - i * dy, str(n_at_risk(times, tq)), transform=ax.transAxes, fontsize=fs, ha='center', va='center')


FU_NEW, FU_REG = 730.0, 1095.0


def surv_frame(R, ids, first_event, fu):
    ec = R['ec']; last_echo = ec[ec['d_dis'] > 0].groupby('科研患者编号')['d_dis'].max()
    d = pd.DataFrame(index=pd.Index([i for i in ids if i in last_echo.index], name='科研患者编号'))
    d['t_last'] = last_echo.reindex(d.index); d['t_ev'] = first_event.reindex(d.index)
    d['event'] = d['t_ev'].notna().astype(int); d['time'] = np.where(d['event'] == 1, d['t_ev'], d['t_last']).astype(float)
    d['time_c'] = np.minimum(d['time'], fu); d['event_c'] = np.where(d['time'] > fu, 0, d['event'])
    return d


def first_post(R, cond):
    ec = R['ec']; q = ec[(ec['d_dis'] > 0) & cond(ec)]
    return q.groupby('科研患者编号')['d_dis'].min()


def tte(R):
    c1 = R['c1'].set_index('科研患者编号'); ec = R['ec']
    out = {}; rows = []

    def record(analysis, group, d, tqs):
        kd = km(d['time_c'], d['event_c'])
        for tq in tqs:
            S, lo, hi = km_at(kd, tq)
            rows.append(dict(scope=R['scope'], analysis=analysis, group=group, n=len(d), events=int(d['event_c'].sum()), time_d=tq, cum_pct=round((1 - S) * 100, 2),
                             ci_low=round((1 - hi) * 100, 2), ci_high=round((1 - lo) * 100, 2), n_at_risk=n_at_risk(d['time_c'], tq)))
        return kd
    # TTE-1
    f_an = first_post(R, lambda x: x['cat'] >= 2)
    u = surv_frame(R, c1.index[c1['in_max'] == 0], f_an, FU_NEW).join(c1[['age_lt12m', 'ivig_resistant', 'inflam_dis']])
    u['risk_c'] = (u[['age_lt12m', 'ivig_resistant']].astype(int).sum(axis=1) + u['inflam_dis'].astype(int)).clip(upper=2).map({0: '0 factors', 1: '1 factor', 2: '≥2 factors'})
    out['u'] = u; out['km_u'] = record('TTE-1 new aneurysm, uncomplicated in hospital', 'all', u, [30, 90, 180, 365, 730])
    out['km_u_risk'] = {g: record('TTE-1 by risk-factor count', g, u[u['risk_c'] == g], [30, 90, 365]) for g in ['0 factors', '1 factor', '≥2 factors']}
    out['lr_u_risk'] = logrank(u['time_c'], u['event_c'], u['risk_c'])
    z25 = surv_frame(R, c1.index[c1['in_max'] <= 1], f_an, FU_NEW).join(c1[['in_max']]); z25['grp'] = z25['in_max'].map({0.0: 'None (Z<2)', 1.0: 'Dilation only (Z 2–<2.5)'})
    out['z25'] = z25; out['km_z25'] = {g: record('TTE-1b new aneurysm by in-hospital category', g, z25[z25['grp'] == g], [30, 90, 365]) for g in ['None (Z<2)', 'Dilation only (Z 2–<2.5)']}
    out['lr_z25'] = logrank(z25['time_c'], z25['event_c'], z25['grp'])
    # TTE-2 regression by size
    f_reg = first_post(R, lambda x: x['cat'] <= 1); f_norm = first_post(R, lambda x: x['cat'] == 0)
    an_ids = c1.index[c1['in_max'] >= 2]
    rg = surv_frame(R, an_ids, f_reg, FU_REG).join(c1[['in_max', 'ivig_resistant']]); rg['size'] = rg['in_max'].map({2.0: 'Small', 3.0: 'Medium', 4.0: 'Large/giant'})
    nm_ = surv_frame(R, an_ids, f_norm, FU_REG).join(c1[['in_max']]); nm_['size'] = nm_['in_max'].map({2.0: 'Small', 3.0: 'Medium', 4.0: 'Large/giant'})
    out['rg'] = rg; out['nm'] = nm_
    out['km_rg'] = {g: record('TTE-2 regression below aneurysm threshold (Z<2.5)', g, rg[rg['size'] == g], [30, 90, 180, 365, 730, 1095]) for g in ['Small', 'Medium', 'Large/giant']}
    out['km_nm'] = {g: record('TTE-2 normal lumen (Z<2)', g, nm_[nm_['size'] == g], [30, 90, 180, 365, 730, 1095]) for g in ['Small', 'Medium', 'Large/giant']}
    out['lr_rg'] = logrank(rg['time_c'], rg['event_c'], rg['size']); out['lr_nm'] = logrank(nm_['time_c'], nm_['event_c'], nm_['size'])
    out['median_reg'] = {g: median_time(k) for g, k in out['km_rg'].items()}; out['median_norm'] = {g: median_time(k) for g, k in out['km_nm'].items()}
    # TTE-3 progression
    inmax = c1['in_max']
    f_prog = ec[(ec['d_dis'] > 0)].assign(im=lambda x: x['科研患者编号'].map(inmax)).query('cat > im').groupby('科研患者编号')['d_dis'].min()
    pg = surv_frame(R, c1.index[c1['in_max'].isin([1.0, 2.0, 3.0])], f_prog, FU_REG).join(c1[['in_max', 'ivig_resistant']])
    pg['start'] = pg['in_max'].map({1.0: 'Dilation only', 2.0: 'Small', 3.0: 'Medium'})
    out['pg'] = pg
    out['km_pg'] = {g: record('TTE-3 progression to a higher category', g, pg[pg['start'] == g], [30, 90, 365]) for g in ['Dilation only', 'Small', 'Medium']}
    pga = pg[pg['start'].isin(['Small', 'Medium'])]
    out['km_pg_ivig'] = {g: record('TTE-3 progression of small/medium aneurysms, by IVIG resistance', 'IVIG-resistant' if g else 'IVIG-responsive', pga[pga['ivig_resistant'] == g], [30, 90, 365]) for g in [True, False]}
    out['lr_pg_ivig'] = logrank(pga['time_c'], pga['event_c'], pga['ivig_resistant'])
    # TTE-4 illness day of first documentation
    fi = {th: ec[ec['cat'] >= th].sort_values('检查时间').groupby('科研患者编号')['ill_day'].first().dropna() for th in (1, 2)}
    out['illday_first'] = fi
    out['illday_pct'] = {f'cat>={th}': {dq: round(float((s <= dq).mean() * 100), 1) for dq in [7, 10, 14, 21, 28, 42, 60, 90, 365]} for th, s in fi.items()}
    out['table'] = pd.DataFrame(rows)
    return out


TTE = {s: tte(RES[s]) for s in ['primary', 'all']}
T6 = pd.concat([TTE[s]['table'] for s in TTE], ignore_index=True); T6.to_csv(os.path.join(OUT_DIR, 'Table6_time_to_event.csv'), index=False)
tp = TTE['primary']
print('TTE-1 (primary): n =', len(tp['u']), 'events =', int(tp['u']['event_c'].sum()), '| log-rank by risk count P =', f"{tp['lr_u_risk'][2]:.2g}", '| dilation only vs none P =', f"{tp['lr_z25'][2]:.2g}")
print('TTE-2 (primary): n =', len(tp['rg']), tp['rg']['size'].value_counts().to_dict(), '| median days to Z<2.5:', tp['median_reg'], '| to Z<2:', tp['median_norm'], '| log-rank P =', f"{tp['lr_rg'][2]:.2g}", f"{tp['lr_nm'][2]:.2g}")
print('TTE-3 (primary): progression by start', tp['pg']['start'].value_counts().to_dict(), '| IVIG log-rank P =', f"{tp['lr_pg_ivig'][2]:.2g}")
print(T6[T6['scope'] == 'primary'].to_string(index=False))

## 8. Figures (primary scope; Figure S2 compares the primary and the all-data scopes)

In [ ]:
def panel_letter(ax, letter, x=-0.13):
    ax.text(x, 1.03, letter, transform=ax.transAxes, fontsize=15, fontweight='bold', va='bottom', ha='left')


def savefig(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(os.path.join(OUT_DIR, f'{name}.{ext}'), bbox_inches='tight', facecolor='white')
    plt.close(fig)


def fmtp(p):
    return 'P < 0.001' if p < 0.001 else f'P = {p:.3f}' if p < 0.1 else f'P = {p:.2f}'


CATC = ['#ffffff', '#d9d9d9', '#a6a6a6', '#636363', '#000000']
seq = P_['seq'].set_index('step')
# ---------- Figure 1: surveillance ladder (A) and where coronary findings are first documented (B) ----------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.9), gridspec_kw={'width_ratios': [1.15, 1]})
ax = axes[0]
steps = [('First echo ≤30 d\nafter discharge', 'First echo ≤30 d after an uncomplicated in-hospital course: aneurysm (Z≥2.5)', '   … any involvement (Z≥2)'),
         ('31–90 d\nafter two normal', '   … aneurysm', '31–90 d after uncomplicated in-hospital + normal ≤30-d echoes: any involvement'),
         ('91–365 d\nafter three normal', None, '91–365 d after normal findings at all three prior windows: any involvement'),
         ('>365 d\nafter four normal', None, '>365 d after normal findings at all four prior windows: any involvement')]
srows = P_['seq'].reset_index(drop=True)
idx_an = [0, 5, 7, 9]; idx_inv = [1, 4, 6, 8]
x = np.arange(4)
for off, ids, col, lab in [(-0.2, idx_inv, LGREY, 'Any involvement (Z ≥ 2)'), (0.2, idx_an, BLACK, 'Aneurysm (Z ≥ 2.5)')]:
    r_ = srows.loc[ids]
    ax.bar(x + off, r_['pct'], 0.38, color=col, edgecolor='black', linewidth=0.5, label=lab)
    ax.errorbar(x + off, r_['pct'], yerr=[r_['pct'] - r_['ci_low'], r_['ci_high'] - r_['pct']], fmt='none', ecolor='black', elinewidth=0.9, capsize=3)
    for xi, (_, rr) in zip(x + off, r_.iterrows()):
        ax.text(xi, rr['ci_high'] + 0.08, f"{int(rr['events'])}/{int(rr['n']):,}", ha='center', va='bottom', fontsize=8, rotation=90)
ax.set_xticks(x); ax.set_xticklabels([s[0] for s in steps], fontsize=9); ax.set_ylabel('New finding (%)'); ax.set_ylim(0, srows.loc[idx_inv + idx_an, 'ci_high'].max() * 1.6)
ax.legend(frameon=False, fontsize=9, loc='upper right'); panel_letter(ax, 'A')
ax = axes[1]
fd = P_['first_doc']; cols_w = [BLACK, '#3a3a3a', DGREY, MGREY, LGREY]
for yi, (key, lab) in enumerate([('cat>=3', 'Medium or larger (Z ≥ 5)'), ('cat>=2', 'Aneurysm (Z ≥ 2.5)'), ('cat>=1', 'Any involvement (Z ≥ 2)')]):
    tab = fd[key]; total = int(tab.sum()); left = 0.0
    for wv, col in zip(WORDER, cols_w):
        v = int(tab[wv]); pct = v / total * 100
        ax.barh(yi, pct, left=left, color=col, edgecolor='black', linewidth=0.4, label=(wv[0].upper() + wv[1:]) if yi == 0 else None)
        if pct >= 5:
            ax.text(left + pct / 2, yi, f'{v}\n({pct:.0f}%)', ha='center', va='center', fontsize=8, color='white' if col in (BLACK, '#3a3a3a', DGREY) else 'black')
        left += pct
    ax.text(101, yi, f'n = {total}', va='center', fontsize=9)
ax.set_yticks([0, 1, 2]); ax.set_yticklabels(['Medium or larger', 'Aneurysm', 'Any involvement'], fontsize=10); ax.set_xlim(0, 118); ax.set_ylim(-0.6, 3.2)
ax.set_xlabel('Children, by window of the first report (%)')
ax.legend(frameon=False, fontsize=8, ncol=3, loc='upper center', title='Window of first report', title_fontsize=8.5); panel_letter(ax, 'B', -0.25)
fig.tight_layout(); savefig(fig, 'Figure1_surveillance_ladder_and_first_documentation')

# ---------- Figure 2: who has an aneurysm at the first post-discharge echo (Z<2.5 in hospital) ----------
fig, axes = plt.subplots(1, 2, figsize=(13, 5.6), gridspec_kw={'width_ratios': [1.1, 1]})
ax = axes[0]
order = ['In-hospital no involvement (Z<2)', 'In-hospital dilation only (Z 2–<2.5)', '<6 mo', '6–11 mo', '12–23 mo', '2–4 y', '≥5 y', 'Female', 'Male', 'IVIG-responsive', 'IVIG-resistant',
         'Discharge WBC ≤12', 'Discharge WBC >12', 'Not readmitted', 'Readmitted ≤30 d', '0 factors', '1 factor', '≥2 factors']
names = {'Discharge WBC >12': 'Discharge WBC >12 ×10⁹/L', 'Discharge WBC ≤12': 'Discharge WBC ≤12 ×10⁹/L', '0 factors': 'Risk factors: 0', '1 factor': 'Risk factors: 1', '≥2 factors': 'Risk factors: ≥2',
         'In-hospital no involvement (Z<2)': 'In hospital: Z < 2', 'In-hospital dilation only (Z 2–<2.5)': 'In hospital: dilation only'}
st = strata.set_index('stratum').loc[order]; yy = np.arange(len(st))[::-1]
ax.errorbar(st['pct'], yy, xerr=[st['pct'] - st['ci_low'], st['ci_high'] - st['pct']], fmt='s', color='black', ecolor='black', elinewidth=1, capsize=2.5, markersize=5)
for yi, (_, r_) in zip(yy, st.iterrows()):
    ax.text(st['ci_high'].max() * 1.04, yi, f"{int(r_['events'])}/{int(r_['n']):,}", va='center', fontsize=8.5)
ax.set_yticks(yy); ax.set_yticklabels([names.get(i, i) for i in st.index], fontsize=9); ax.set_xlabel('Aneurysm at first post-discharge echo (%)'); ax.set_xlim(0, st['ci_high'].max() * 1.28)
for sep in [15.5, 10.5, 8.5, 6.5, 4.5, 2.5]:
    ax.axhline(sep, color=LGREY, lw=0.6)
panel_letter(ax, 'A', -0.42)
ax = axes[1]
mvb = MB.copy(); y2 = np.arange(len(mvb))[::-1]
ax.errorbar(mvb['OR'], y2, xerr=[mvb['OR'] - mvb['CI_low'], mvb['CI_high'] - mvb['OR']], fmt='o', color='black', ecolor='black', elinewidth=1, capsize=2.5, markersize=5)
ax.axvline(1, color=MGREY, lw=0.8, linestyle='--', label='Odds ratio = 1')
ax.set_xscale('log'); ax.set_yticks(y2); ax.set_yticklabels(mvb['variable'], fontsize=9); ax.set_xlabel('Adjusted odds ratio (Firth), log scale')
for yi, (_, r_) in zip(y2, mvb.iterrows()):
    ax.text(28, yi, f"{r_['OR']:.2f} ({r_['CI_low']:.2f}–{r_['CI_high']:.2f})", va='center', fontsize=8.5)
ax.set_xlim(0.3, 80); ax.set_xticks([0.5, 1, 2, 5, 10, 20]); ax.set_xticklabels(['0.5', '1', '2', '5', '10', '20']); ax.minorticks_off()
ax.legend(frameon=False, fontsize=8.5, loc='lower right', bbox_to_anchor=(1.0, 1.0)); panel_letter(ax, 'B', -0.75)
fig.tight_layout(); savefig(fig, 'Figure2_risk_of_aneurysm_at_first_postdischarge_echo')

# ---------- Figure 3: course of in-hospital aneurysms by size ----------
fig, axes = plt.subplots(1, 3, figsize=(17, 5.8)); plt.subplots_adjust(bottom=0.33, wspace=0.42)
tq3 = [0, 180, 365, 730, 1095]; scol = {'Small': LGREY, 'Medium': MGREY, 'Large/giant': BLACK}
for ax, key, frame, ylab, lrk, letter in [(axes[0], 'km_rg', 'rg', 'Aneurysm still present, Z ≥ 2.5 (%)', 'lr_rg', 'A'), (axes[1], 'km_nm', 'nm', 'Lumen not yet normal, Z ≥ 2 (%)', 'lr_nm', 'B')]:
    for g in ['Small', 'Medium', 'Large/giant']:
        plot_km(ax, tp[key][g], f"{g} (n = {int((tp[frame]['size'] == g).sum())})", scol[g])
    ax.plot([], [], ' ', label=f'Log-rank {fmtp(tp[lrk][2])}')
    ax.set_xlim(0, FU_REG); ax.set_xticks(tq3); ax.set_ylim(0, 100); ax.set_xlabel('Days after discharge'); ax.set_ylabel(ylab)
    ax.legend(frameon=False, fontsize=8, loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=2, title='In-hospital maximum', title_fontsize=8)
    risk_table(ax, [tp[frame].loc[tp[frame]['size'] == g, 'time_c'] for g in ['Small', 'Medium', 'Large/giant']], ['Small', 'Medium', 'Large/giant'], tq3)
    panel_letter(ax, letter, -0.2)
ax = axes[2]; tq3c = [0, 90, 180, 365]
for g, col in [('Dilation only', LGREY), ('Small', MGREY), ('Medium', BLACK)]:
    plot_km(ax, tp['km_pg'][g], f"{g} (n = {int((tp['pg']['start'] == g).sum())})", col, cumulative=True)
ax.set_xlim(0, 365); ax.set_xticks(tq3c); ax.set_ylim(0, 40); ax.set_xlabel('Days after discharge'); ax.set_ylabel('Progression to a higher category (%)')
ax.legend(frameon=False, fontsize=8, loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=2, title='In-hospital maximum', title_fontsize=8)
risk_table(ax, [tp['pg'].loc[tp['pg']['start'] == g, 'time_c'] for g in ['Dilation only', 'Small', 'Medium']], ['Dilation only', 'Small', 'Medium'], tq3c)
panel_letter(ax, 'C', -0.2)
savefig(fig, 'Figure3_course_of_inhospital_findings_by_size')

# ---------- Figure 4: new aneurysm after discharge ----------
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.6)); plt.subplots_adjust(bottom=0.34, wspace=0.35)
tq4 = [0, 30, 90, 180, 365]
ax = axes[0]
for g, col in [('0 factors', LGREY), ('1 factor', DGREY), ('≥2 factors', BLACK)]:
    plot_km(ax, tp['km_u_risk'][g], f"{g} (n = {int((tp['u']['risk_c'] == g).sum()):,})", col, cumulative=True)
ax.plot([], [], ' ', label=f"Log-rank {fmtp(tp['lr_u_risk'][2])}")
ax.set_xlim(0, 365); ax.set_xticks(tq4); ax.set_ylim(0, 10); ax.set_xlabel('Days after discharge'); ax.set_ylabel('New aneurysm, cumulative incidence (%)')
ax.legend(frameon=False, fontsize=8.5, loc='upper left', title='Z < 2 in hospital; risk factors', title_fontsize=8.5)
risk_table(ax, [tp['u'].loc[tp['u']['risk_c'] == g, 'time_c'] for g in ['0 factors', '1 factor', '≥2 factors']], ['0 factors', '1 factor', '≥2 factors'], [0, 90, 180, 365])
panel_letter(ax, 'A', -0.2)
ax = axes[1]
for g, col in [('None (Z<2)', LGREY), ('Dilation only (Z 2–<2.5)', BLACK)]:
    plot_km(ax, tp['km_z25'][g], f"{g} (n = {int((tp['z25']['grp'] == g).sum()):,})", col, cumulative=True)
ax.plot([], [], ' ', label=f"Log-rank {fmtp(tp['lr_z25'][2])}")
ax.set_xlim(0, 365); ax.set_xticks(tq4); ax.set_ylim(0, 20); ax.set_yticks([0, 5, 10, 15, 20]); ax.set_xlabel('Days after discharge'); ax.set_ylabel('New aneurysm, cumulative incidence (%)')
ax.legend(frameon=False, fontsize=8.5, loc='upper left', title='In-hospital category', title_fontsize=8.5)
risk_table(ax, [tp['z25'].loc[tp['z25']['grp'] == g, 'time_c'] for g in ['None (Z<2)', 'Dilation only (Z 2–<2.5)']], ['Z < 2', 'Dilation only'], [0, 90, 180, 365])
panel_letter(ax, 'B', -0.2)
savefig(fig, 'Figure4_new_aneurysm_after_discharge')

# ---------- Figure 5: AHA status by illness-day window ----------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for ax, tab, letter in [(axes[0], P_['illday'], 'A'), (axes[1], P_['illday_uncomplicated'], 'B')]:
    xx = np.arange(len(tab))
    ax.bar(xx, tab['involvement_pct'], color=LGREY, edgecolor='black', linewidth=0.4, label='Any involvement (Z ≥ 2)')
    ax.bar(xx, tab['aneurysm_pct'], color=BLACK, edgecolor='black', linewidth=0.4, label='Aneurysm (Z ≥ 2.5)')
    for i, v in enumerate(tab['involvement_pct']):
        ax.text(i, v + 0.3, f'{v:.1f}', ha='center', va='bottom', fontsize=7.5)
    ax.set_xticks(xx); ax.set_xticklabels([f'{w}\n{n:,}' for w, n in zip(tab.index, tab['children'])], fontsize=7.5)
    ax.set_xlabel('Illness-day window (children with an echo in the window)'); ax.set_ylabel('Children (%)'); ax.set_ylim(0, P_['illday']['involvement_pct'].max() * 1.2)
    ax.legend(frameon=False, fontsize=8.5, loc='upper right'); panel_letter(ax, letter, -0.1)
fig.tight_layout(); savefig(fig, 'Figure5_status_by_illness_day')

# ---------- Figure 6: illness day of first documentation ----------
fig, ax = plt.subplots(figsize=(7, 4.6))
for th, col, lab in [(1, DGREY, 'Any involvement (Z ≥ 2)'), (2, BLACK, 'Aneurysm (Z ≥ 2.5)')]:
    ser = tp['illday_first'][th]; xs_ = np.sort(ser.values); ys_ = np.arange(1, len(xs_) + 1) / len(xs_) * 100
    ax.step(np.concatenate([[0], xs_]), np.concatenate([[0], ys_]), where='post', color=col, lw=1.7, label=f'{lab} (n = {len(ser)})')
for dq, ls in [(14, ':'), (42, '--')]:
    ax.axvline(dq, color='black', lw=0.8, linestyle=ls, label=f'Illness day {dq}')
ax.set_xscale('log'); ax.set_xlim(1, 2000); ax.set_xticks([1, 3, 7, 14, 28, 60, 120, 365, 1000]); ax.set_xticklabels(['1', '3', '7', '14', '28', '60', '120', '365', '1000']); ax.minorticks_off()
ax.set_xlabel('Illness day of the first report (log scale)'); ax.set_ylabel('First documented by this day (%)'); ax.set_ylim(0, 100); ax.legend(frameon=False, fontsize=9, loc='lower right')
savefig(fig, 'Figure6_illness_day_of_first_documentation')
print('main figures written')

## 9. Plan B: primary (numeric era) vs all-data estimates; cohort flow; cross-check with the frozen master (IVIG resistance, fever days)

In [ ]:
def km_row(tbl, scope, analysis, group, tq):
    r_ = tbl[(tbl['scope'] == scope) & (tbl['analysis'] == analysis) & (tbl['group'] == group) & (tbl['time_d'] == tq)]
    return (float(r_['cum_pct'].iloc[0]), float(r_['ci_low'].iloc[0]), float(r_['ci_high'].iloc[0]), int(r_['n'].iloc[0])) if len(r_) else (np.nan, np.nan, np.nan, 0)


def dil_only_rate(R):
    c1_ = R['c1']; dz = c1_[(c1_['in_max'] == 1) & (c1_['post30_n'] > 0)]
    return rate(int((dz['post30_cat_first'] >= 2).sum()), len(dz))


cmp_rows = []
for s in ['primary', 'all']:
    R = RES[s]; sq = R['seq'].reset_index(drop=True)
    def add_cmp(name, est, lo, hi, n, extra=''):
        cmp_rows.append(dict(scope=s, estimate=name, value=est, ci_low=lo, ci_high=hi, n=n, note=extra))
    r_ = R['inhosp_aneurysm']; add_cmp('In-hospital aneurysm (Z ≥ 2.5), % of cohort C1', r_['pct'], r_['ci_low'], r_['ci_high'], r_['n'])
    r_ = sq.loc[0]; add_cmp('Aneurysm at the first echo ≤30 d after an uncomplicated course, %', r_['pct'], r_['ci_low'], r_['ci_high'], r_['n'],
                            f"probability-weighted events {r_['prob_weighted_events']}")
    r_ = sq.loc[1]; add_cmp('Any involvement at the first echo ≤30 d after an uncomplicated course, %', r_['pct'], r_['ci_low'], r_['ci_high'], r_['n'],
                            f"probability-weighted events {r_['prob_weighted_events']}")
    r_ = sq.loc[4]; add_cmp('Any involvement at 31–90 d after two normal windows, %', r_['pct'], r_['ci_low'], r_['ci_high'], r_['n'])
    r_ = dil_only_rate(R); add_cmp('Aneurysm at the first echo ≤30 d after in-hospital dilation only, %', r_['pct'], r_['ci_low'], r_['ci_high'], r_['n'])
    k_ = R['aneurysm_first_after_discharge']; n_ = R['aneurysm_ever']; lo, hi = wilson(k_, n_)
    add_cmp('Aneurysms first documented after discharge, % of all aneurysms', round(k_ / n_ * 100, 2), round(lo, 2), round(hi, 2), n_)
    for g in ['Small', 'Medium', 'Large/giant']:
        v = km_row(T6, s, 'TTE-2 regression below aneurysm threshold (Z<2.5)', g, 365); add_cmp(f'Regression below Z 2.5 by 1 year, {g.lower()} aneurysm, %', *v)
    for g in ['Small', 'Medium']:
        v = km_row(T6, s, 'TTE-3 progression to a higher category', g, 365); add_cmp(f'Progression to a larger category by 1 year, {g.lower()} aneurysm, %', *v)
    v = km_row(T6, s, 'TTE-1 new aneurysm, uncomplicated in hospital', 'all', 365); add_cmp('New aneurysm by 1 year after an uncomplicated course (cumulative incidence), %', *v)
CMP = pd.DataFrame(cmp_rows)
CMPw = CMP.pivot(index='estimate', columns='scope', values=['value', 'ci_low', 'ci_high', 'n'])
CMPw.columns = [f'{a}_{b}' for a, b in CMPw.columns]
CMPw = CMPw.loc[CMP['estimate'].drop_duplicates().tolist(), [f'{a}_{b}' for b in ['primary', 'all'] for a in ['value', 'ci_low', 'ci_high', 'n']]]
CMPw.to_csv(os.path.join(OUT_DIR, 'Table7_planB_primary_vs_alldata.csv')); print(CMPw.round(2).to_string())

# Figure S2 - Plan B comparison
fig, ax = plt.subplots(figsize=(10.5, 6.6))
ests = CMPw.index.tolist(); yy = np.arange(len(ests))[::-1]
for off, s, mk, col, lab in [(0.15, 'primary', 's', BLACK, 'Primary: 2018–2024, echoes to the end of the numeric era'), (-0.15, 'all', 'o', MGREY, 'All data: 2018–2026, wording model for table-only reports')]:
    v = CMPw[f'value_{s}'].values; lo = CMPw[f'ci_low_{s}'].values; hi = CMPw[f'ci_high_{s}'].values
    ax.errorbar(v, yy + off, xerr=[v - lo, hi - v], fmt=mk, color=col, ecolor=col, elinewidth=1, capsize=2.5, markersize=5, label=lab)
ax.set_yticks(yy); ax.set_yticklabels(ests, fontsize=8.5); ax.set_xscale('symlog', linthresh=1); ax.set_xlim(0, 110)
ax.set_xticks([0, 0.5, 1, 2, 5, 10, 20, 50, 100]); ax.set_xticklabels(['0', '0.5', '1', '2', '5', '10', '20', '50', '100'])
ax.set_xlabel('Estimate (%; 95% CI; symmetric-log scale)'); ax.legend(frameon=False, fontsize=8.5, loc='lower center', bbox_to_anchor=(0.4, 1.0), ncol=1)
savefig(fig, 'FigureS2_planB_primary_vs_alldata')

# Figure S1 - cohort flow (primary scope)
from matplotlib.patches import FancyBboxPatch
mmP = SC['primary'][0]
F = dict(all_index=len(m), y2018=int((m['year'] >= 2018).sum()), y2018_24=int(((m['year'] >= 2018) & (m['year'] <= 2024)).sum()), excl2021=int((m['year'] == 2021).sum()),
         primary=len(mmP), c1=P_['C1_n'], no_echo=int(len(mmP) - P_['C1_n']), y2025_26=int((m['year'] >= 2025).sum()))
fig, ax = plt.subplots(figsize=(9.5, 10.5)); ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
def box(x, y, w, h, t, fs=9.2, bold=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.008,rounding_size=0.01', linewidth=1, edgecolor='black', facecolor='white'))
    ax.text(x + w / 2, y + h / 2, t, ha='center', va='center', fontsize=fs, fontweight='bold' if bold else 'normal')
def arrow(x0, y0, x1, y1):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle='->', lw=1, color='black'))
box(0.03, 0.90, 0.5, 0.07, f"First inpatient Kawasaki disease diagnosis,\n2012–{int(m['year'].max())}, n = {F['all_index']:,}")
box(0.03, 0.77, 0.5, 0.07, f"Index admission 2018–2024, not 2021\nn = {F['primary']:,}")
box(0.58, 0.835, 0.4, 0.075, f"Excluded: before 2018 n = {F['all_index'] - F['y2018']:,};\n2021 (in-hospital reports missing) n = {F['excl2021']:,};\n2025–2026 (table-only era; sensitivity analysis) n = {F['y2025_26']:,}", fs=8.2)
box(0.03, 0.64, 0.5, 0.075, f"Cohort C1: ≥1 classifiable in-hospital echo\nn = {F['c1']:,} (echoes to {ERA_END.date()})", bold=True)
box(0.58, 0.70, 0.4, 0.05, f"Excluded: no in-hospital echo report, n = {F['no_echo']:,}", fs=8.2)
arrow(0.28, 0.90, 0.28, 0.84); arrow(0.28, 0.77, 0.28, 0.715); arrow(0.28, 0.87, 0.58, 0.87); arrow(0.28, 0.725, 0.58, 0.725)
xs = [0.01, 0.208, 0.406, 0.604, 0.802]; ws_ = 0.185
for k, x_ in enumerate(xs):
    box(x_, 0.47, ws_, 0.08, f"{CATL[k].replace(' (', chr(10) + '(')}\nn = {P_['inhosp_cat'].get(CATS[k], 0):,}", fs=7.8)
    arrow(0.28, 0.64, x_ + ws_ / 2, 0.55)
coh_ = P_['coh']
box(0.01, 0.25, 0.37, 0.13, f"Uncomplicated (Z < 2): first echo ≤30 d\nn = {P_['surv_n']:,} (no echo ≤30 d: {P_['no_echo_30d']:,})\n→ aneurysm {int(coh_['new_an'].sum())} ({coh_['new_an'].mean()*100:.2f}%);\nany involvement {int(coh_['new_inv'].sum())} ({coh_['new_inv'].mean()*100:.2f}%)", fs=8.4, bold=True)
dz_ = dil_only_rate(P_)
box(0.40, 0.25, 0.2, 0.13, f"Dilation only:\nfirst echo ≤30 d\nn = {dz_['n']}\n→ aneurysm {dz_['events']}\n({dz_['pct']:.1f}%)", fs=8.4)
rgP = tp['rg']
box(0.62, 0.25, 0.37, 0.13, f"In-hospital aneurysm with follow-up\nn = {len(rgP)} (small {int((rgP['size'] == 'Small').sum())}, medium {int((rgP['size'] == 'Medium').sum())},\nlarge/giant {int((rgP['size'] == 'Large/giant').sum())}) → regression and\nprogression (Figure 3)", fs=8.4)
arrow(xs[0] + ws_ / 2, 0.47, 0.195, 0.38); arrow(xs[1] + ws_ / 2, 0.47, 0.50, 0.38)
for k in (2, 3, 4):
    arrow(xs[k] + ws_ / 2, 0.47, 0.805, 0.38)
box(0.01, 0.12, 0.98, 0.07, f"Aneurysms first documented after discharge: {P_['aneurysm_first_after_discharge']} of {P_['aneurysm_ever']} children with an aneurysm at any time "
    f"({P_['aneurysm_first_after_discharge']/P_['aneurysm_ever']*100:.1f}%);\nin-hospital category of these children: " +
    ', '.join(f"{k} {v}" for k, v in P_['aneurysm_first_after_discharge_by_inhosp'].items()), fs=8.6)
ax.set_ylim(0.08, 1)
savefig(fig, 'FigureS1_cohort_flow')

# frozen master: IVIG resistance and fever days (coronary crosswalk is in KD_coronary_AHA_v1)
E = {}
MASTER = os.path.join(DATA_DIR, 'KD_Infant_Master_v1.3.xlsx')


def kappa(a, b):
    a = np.asarray(a).astype(int); b = np.asarray(b).astype(int); po = np.mean(a == b)
    pe = sum(np.mean(a == k) * np.mean(b == k) for k in np.union1d(a, b))
    return (po - pe) / (1 - pe) if pe < 1 else np.nan


if os.path.exists(MASTER):
    fm = pd.read_excel(MASTER, sheet_name=0, usecols=['Patient_ID', 'Target_IVIG_Res', 'Fever_Days_On_Admit'])
    fm['科研患者编号'] = fm['Patient_ID'].astype(str)
    x_ = P_['c1'].merge(fm, on='科研患者编号', how='inner')
    xi = x_[x_['Target_IVIG_Res'].notna()]
    E['overlap'] = int(len(x_)); E['ivig_res_n'] = int(len(xi))
    E['ivig_res_agreement_pct'] = round(float((xi['ivig_resistant'].astype(int) == xi['Target_IVIG_Res'].astype(int)).mean() * 100), 1)
    E['ivig_res_kappa'] = round(float(kappa(xi['ivig_resistant'], xi['Target_IVIG_Res'])), 3)
    xf = x_[x_['Fever_Days_On_Admit'].notna() & x_['fever_days_adm'].notna()]
    E['fever_days_n'] = int(len(xf)); E['fever_days_exact_pct'] = round(float((xf['Fever_Days_On_Admit'] == xf['fever_days_adm']).mean() * 100), 1)
    print(E)
else:
    print('frozen master not found — cross-check skipped')

## 10. Results lock and exports

In [ ]:
c1P, cohP = P_['c1'], P_['coh']
# fate of the aneurysms found at the first <=30-d echo after an uncomplicated course
na = cohP[cohP['new_an']]
later = P_['ec'][(P_['ec']['d_dis'] > 30) & P_['ec']['科研患者编号'].isin(na['科研患者编号'])].groupby('科研患者编号')['cat'].agg(['max', 'last'])
FATE = dict(n=int(len(na)), with_later_echo=int(len(later)), last_below_2_5=int((later['last'] <= 1).sum()), later_max_medium_plus=int((later['max'] >= 3).sum()),
            category_at_detection={CATS[int(k)]: int(v) for k, v in na['post30_cat_first'].value_counts().sort_index().items()},
            exact_evidence_at_detection=int((~na['post30_tier4_any'].astype(bool)).sum()))
wx = P_['inhosp_wording_x_cat']
mvb = MB.set_index('variable'); stx = strata.set_index('stratum'); sqP = P_['seq'].reset_index(drop=True)
fdP = P_['first_doc']; ill = tp['illday_pct']
reg = lambda g, tq, an='TTE-2 regression below aneurysm threshold (Z<2.5)': km_row(T6, 'primary', an, g, tq)
fmt_km = lambda v: f"{v[0]:.1f}% ({v[1]:.1f}–{v[2]:.1f})"
L_ = []
L_.append(f"RESULTS LOCK — v4 ({pd.Timestamp.today().date()}) — KD serial-echo surveillance under AHA 2017 coronary categories, Wuhan Children's Hospital")
L_.append(f"Computed by KD_echo_surveillance_v4.ipynb from 川崎病重新导出30天_1/2/3.xlsx (data end {DATA_END.date()}) and the per-report AHA categories of KD_coronary_AHA_v1.ipynb. Every number is computed.\n")
L_.append('R0. Definitions and cohort')
L_.append(f"- AHA 2017 category per report: none Z<2; dilation only 2–<2.5; small 2.5–<5; medium 5–<10; large/giant ≥10 or ≥8 mm. Involvement = ≥ dilation only; aneurysm = ≥ small.")
L_.append(f"- Numeric era ends {ERA_END.date()} (first month with <90% of dilation/aneurysm-worded reports carrying numbers: {first_table_month}). Primary scope = index admissions 2018–2024 (2021 excluded) with echoes to {ERA_END.date()}; within it {((SC['primary'][1]['w'] >= 1) & (SC['primary'][1]['tier'] == 4)).sum()} of {(SC['primary'][1]['w'] >= 1).sum()} dilation/aneurysm-worded reports rely on the wording model.")
L_.append(f"- Cohort C1 (≥1 classifiable in-hospital echo): n = {P_['C1_n']:,}. In-hospital maximum: " + ', '.join(f"{k} {v:,} ({v/P_['C1_n']*100:.1f}%)" for k, v in P_['inhosp_cat'].items()) + f"; aneurysm {P_['inhosp_aneurysm']['events']} ({P_['inhosp_aneurysm']['pct']:.1f}%; 95% CI {P_['inhosp_aneurysm']['ci_low']:.1f}–{P_['inhosp_aneurysm']['ci_high']:.1f}).")
dil_w = wx.loc['dilation'] if 'dilation' in wx.index else None
if dil_w is not None:
    L_.append(f"- In-hospital wording 'dilation' (n = {int(dil_w.sum())}) → AHA: " + ', '.join(f"{c} {int(dil_w.get(c, 0))}" for c in CATS) + f"; wording 'aneurysm' (n = {int(wx.loc['aneurysm'].sum())}) → " + ', '.join(f"{c} {int(wx.loc['aneurysm'].get(c, 0))}" for c in CATS) + '.')
L_.append('R1. Where coronary findings are first documented (C1, primary scope)')
for key, lab in [('cat>=1', 'involvement'), ('cat>=2', 'aneurysm'), ('cat>=3', 'medium or larger')]:
    t_ = fdP[key]; L_.append(f"- {lab}: {int(t_.sum())} children; first in hospital {int(t_.iloc[0])} ({t_.iloc[0]/t_.sum()*100:.1f}%); after discharge " + ', '.join(f"{w} {int(t_[w])}" for w in WORDER[1:]) + '.')
L_.append(f"- Aneurysms first documented after discharge: {P_['aneurysm_first_after_discharge']} of {P_['aneurysm_ever']} ({P_['aneurysm_first_after_discharge']/P_['aneurysm_ever']*100:.1f}%); in-hospital category " +
          ', '.join(f"{k} {v}" for k, v in P_['aneurysm_first_after_discharge_by_inhosp'].items()) + '; per 1,000 followed children: ' +
          '; '.join(f"{r['in_hospital']} {r['per_1000']} (95% CI {r['ci_low']}–{r['ci_high']}; {r['post_discharge_first_aneurysm']}/{r['followed']})" for _, r in P_['pd_first_rates'].iterrows()) + '.')
L_.append('- Higher category after discharge than the in-hospital maximum (followed children): ' + '; '.join(f"{r['in_hospital']} {int(r['higher_category_after_discharge'])}/{int(r['followed'])}" for _, r in P_['worsening'].iterrows()) + '.')
L_.append('R2. Sequential yields after an uncomplicated in-hospital course (Z < 2 on every in-hospital report)')
L_.append(f"- Uncomplicated n = {P_['uncomplicated_n']:,}; with a first echo ≤30 d n = {P_['surv_n']:,} (median day {desc['post30_day_median']:.0f} after discharge [IQR {desc['post30_day_IQR'][0]:.0f}–{desc['post30_day_IQR'][1]:.0f}], illness day {desc['post30_illday_median']:.0f} [{desc['post30_illday_IQR'][0]:.0f}–{desc['post30_illday_IQR'][1]:.0f}]); no echo ≤30 d n = {P_['no_echo_30d']:,} (Table S1).")
for i, r_ in sqP.iterrows():
    extra = f"; probability-weighted events {r_['prob_weighted_events']}" if pd.notna(r_.get('prob_weighted_events', np.nan)) else ''
    L_.append(f"- {r_['step'].strip()}: {int(r_['events'])}/{int(r_['n']):,} = {r_['pct']:.2f}% (95% CI {r_['ci_low']:.2f}–{r_['ci_high']:.2f}){extra}.")
L_.append(f"- Aneurysms found at the first ≤30-d echo (n = {FATE['n']}): category {FATE['category_at_detection']}; exact numeric evidence at detection {FATE['exact_evidence_at_detection']}; re-imaged after day 30 {FATE['with_later_echo']}, last report Z < 2.5 in {FATE['last_below_2_5']}, medium or larger at any later echo {FATE['later_max_medium_plus']}.")
L_.append('R3. Who has an aneurysm at the first post-discharge echo (children discharged with Z < 2.5; Table 4, Figure 2)')
L_.append(f"- n = {nB:,}, {evB} aneurysms. In-hospital dilation only {stx.loc['In-hospital dilation only (Z 2–<2.5)', 'events']}/{stx.loc['In-hospital dilation only (Z 2–<2.5)', 'n']} ({stx.loc['In-hospital dilation only (Z 2–<2.5)', 'pct']:.2f}%) vs Z < 2 {stx.loc['In-hospital no involvement (Z<2)', 'events']}/{stx.loc['In-hospital no involvement (Z<2)', 'n']} ({stx.loc['In-hospital no involvement (Z<2)', 'pct']:.2f}%).")
L_.append(f"- IVIG-resistant {stx.loc['IVIG-resistant', 'pct']:.2f}% vs responsive {stx.loc['IVIG-responsive', 'pct']:.2f}%; discharge WBC >12 {stx.loc['Discharge WBC >12', 'pct']:.2f}% vs ≤12 {stx.loc['Discharge WBC ≤12', 'pct']:.2f}%; readmitted ≤30 d {stx.loc['Readmitted ≤30 d', 'pct']:.2f}% vs {stx.loc['Not readmitted', 'pct']:.2f}%; age <6 mo {stx.loc['<6 mo', 'pct']:.2f}%, ≥5 y {stx.loc['≥5 y', 'pct']:.2f}%.")
L_.append(f"- Risk-factor count (age <12 mo, IVIG resistance, in-hospital dilation only, discharge CRP >20 or WBC >12): 0 → {stx.loc['0 factors', 'events']}/{stx.loc['0 factors', 'n']} ({stx.loc['0 factors', 'pct']:.2f}%); 1 → {stx.loc['1 factor', 'events']}/{stx.loc['1 factor', 'n']} ({stx.loc['1 factor', 'pct']:.2f}%); ≥2 → {stx.loc['≥2 factors', 'events']}/{stx.loc['≥2 factors', 'n']} ({stx.loc['≥2 factors', 'pct']:.2f}%).")
L_.append('- Firth multivariable OR (95% CI): ' + '; '.join(f"{v} {r['OR']:.2f} ({r['CI_low']:.2f}–{r['CI_high']:.2f})" for v, r in mvb.iterrows()) + '.')
L_.append('R4. Course of in-hospital aneurysms by size (Figure 3; time 0 = discharge; Kaplan–Meier)')
for g in ['Small', 'Medium', 'Large/giant']:
    L_.append(f"- {g} (n = {int((tp['rg']['size'] == g).sum())}): below Z 2.5 by 90 d {fmt_km(reg(g, 90))}, 1 y {fmt_km(reg(g, 365))}, 2 y {fmt_km(reg(g, 730))}; normal lumen (Z < 2) by 1 y {fmt_km(reg(g, 365, 'TTE-2 normal lumen (Z<2)'))}, 2 y {fmt_km(reg(g, 730, 'TTE-2 normal lumen (Z<2)'))}; median days to Z < 2.5 {tp['median_reg'][g]}.")
L_.append(f"- Log-rank across sizes: Z < 2.5 {fmtp(tp['lr_rg'][2])}; Z < 2 {fmtp(tp['lr_nm'][2])}.")
for g in ['Dilation only', 'Small', 'Medium']:
    v = km_row(T6, 'primary', 'TTE-3 progression to a higher category', g, 365); L_.append(f"- Progression to a higher category by 1 y, in-hospital {g.lower()} (n = {v[3]}): {fmt_km(v)}.")
L_.append('- Progression of small/medium aneurysms by 1 y: ' + '; '.join(f"{g} {fmt_km(km_row(T6, 'primary', 'TTE-3 progression of small/medium aneurysms, by IVIG resistance', g, 365))}" for g in ['IVIG-resistant', 'IVIG-responsive']) + f"; log-rank {fmtp(tp['lr_pg_ivig'][2])}.")
L_.append('R5. New aneurysm after an uncomplicated course, cumulative incidence (Figure 4)')
L_.append('- ' + '; '.join(f"{t} d {fmt_km(km_row(T6, 'primary', 'TTE-1 new aneurysm, uncomplicated in hospital', 'all', t))}" for t in [30, 90, 365, 730]) + f" (n = {len(tp['u']):,}, {int(tp['u']['event_c'].sum())} events).")
L_.append('- At 1 y by risk-factor count (age <12 mo, IVIG resistance, discharge CRP >20 or WBC >12): ' + '; '.join(f"{g} {fmt_km(km_row(T6, 'primary', 'TTE-1 by risk-factor count', g, 365))}" for g in ['0 factors', '1 factor', '≥2 factors']) + f"; log-rank {fmtp(tp['lr_u_risk'][2])}. In-hospital dilation only vs Z < 2 at 1 y: " +
          '; '.join(f"{g} {fmt_km(km_row(T6, 'primary', 'TTE-1b new aneurysm by in-hospital category', g, 365))}" for g in ['Dilation only (Z 2–<2.5)', 'None (Z<2)']) + f"; log-rank {fmtp(tp['lr_z25'][2])}.")
L_.append('R6. Timing (whole C1, children with a parsable fever onset; Figures 5–6)')
L_.append(f"- First documented involvement (n = {len(tp['illday_first'][1])}) by illness day 10 {ill['cat>=1'][10]}%, 14 {ill['cat>=1'][14]}%, 21 {ill['cat>=1'][21]}%, 42 {ill['cat>=1'][42]}%; aneurysm (n = {len(tp['illday_first'][2])}) by day 10 {ill['cat>=2'][10]}%, 14 {ill['cat>=2'][14]}%, 21 {ill['cat>=2'][21]}%, 42 {ill['cat>=2'][42]}%, 90 {ill['cat>=2'][90]}%.")
L_.append('- Involvement by illness-day window (%; children with an echo): ' + ', '.join(f"{w} {r['involvement_pct']:.1f} (aneurysm {r['aneurysm_pct']:.1f}; n = {int(r['children']):,})" for w, r in P_['illday'].iterrows()) + '.')
L_.append('R7. Plan B — primary vs all-data estimates (Table 7, Figure S2)')
for est, r_ in CMPw.iterrows():
    L_.append(f"- {est}: primary {r_['value_primary']:.2f} ({r_['ci_low_primary']:.2f}–{r_['ci_high_primary']:.2f}; n = {int(r_['n_primary'])}) | all data {r_['value_all']:.2f} ({r_['ci_low_all']:.2f}–{r_['ci_high_all']:.2f}; n = {int(r_['n_all'])}).")
L_.append('- Sensitivity analyses of the first-echo yield (Table S2): ' + '; '.join(f"{r['analysis'].split(':')[0]} {int(r['events'])}/{int(r['n'])} ({r['pct']:.2f}%)" for _, r in tS2.iterrows()) + '.')
if E:
    L_.append('R8. Cross-check with the frozen master (reference only)')
    L_.append(f"- IVIG resistance (medication orders vs Target_IVIG_Res): agreement {E['ivig_res_agreement_pct']}%, kappa {E['ivig_res_kappa']} (n = {E['ivig_res_n']:,}); fever days at admission exact agreement {E['fever_days_exact_pct']}% (n = {E['fever_days_n']:,}).")
L_.append("\nCaveats: (1) 2021 in-hospital reports missing; (2) from March 2025 the coronary measurements sit in an unexported table — the primary scope stops at the end of the numeric era and the all-data scope uses wording-model categories; (3) events are dated at the reporting echo (interval-censored in reality); time 0 = discharge, so regression partly reflects the timing of the first follow-up echo; (4) children without a ≤30-d echo in our system are not in the yield denominators (Table S1); (5) fever onset parsed for ~76%.")
lock = '\n'.join(L_)
open(os.path.join(OUT_DIR, 'Results_locked_v4.txt'), 'w', encoding='utf-8').write(lock); print(lock)
KEYv4 = dict(era_end=str(ERA_END.date()), primary={k: v for k, v in P_.items() if isinstance(v, (int, float, dict, str))}, all={k: v for k, v in S_.items() if isinstance(v, (int, float, dict, str))},
             firth_model=MB.to_dict('records'), fate=FATE, planB=CMP.to_dict('records'), master=E)
json.dump(KEYv4, open(os.path.join(OUT_DIR, 'key_numbers.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=str)
keep = ['科研患者编号', 'index_enc', 'year', 'age_m', 'male', 'ivig_given', 'ivig_resistant', 'readm30', 'in_index_n', 'in_max', 'in_index_w_max', 'in_index_z_max',
        'post30_n', 'post30_cat_first', 'post30_cat_max', 'post30_day_first', 'post30_illday_first', 'd31_90_cat_max', 'd91_365_cat_max', 'gt365_cat_max']
c1P[keep].to_csv(os.path.join(OUT_DIR, 'cohort_C1_primary.csv'), index=False)
# tables behind R1, R2, R4 and R6 (both scopes; every cell computed above)
def stack_scopes(key, index_name=None):
    out = []
    for s in ['primary', 'all']:
        t = RES[s][key]
        t = t.rename_axis(index_name).reset_index() if index_name else t.copy()
        t.insert(0, 'scope', s); out.append(t)
    return pd.concat(out, ignore_index=True)


stack_scopes('seq').to_csv(os.path.join(OUT_DIR, 'Table2_sequential_yield.csv'), index=False)
t3a = stack_scopes('first_doc', 'first_documented_in').rename(columns={'cat>=1': 'involvement_Z2', 'cat>=2': 'aneurysm_Z2.5', 'cat>=3': 'medium_or_larger_Z5'})
for c in ['involvement_Z2', 'aneurysm_Z2.5', 'medium_or_larger_Z5']:
    t3a[c + '_pct'] = (t3a[c] / t3a.groupby('scope')[c].transform('sum') * 100).round(1)
t3a.to_csv(os.path.join(OUT_DIR, 'Table3a_first_documentation_by_window.csv'), index=False)
t3b = stack_scopes('worsening').merge(stack_scopes('pd_first_rates'), on=['scope', 'in_hospital', 'followed'], how='left')
t3b.to_csv(os.path.join(OUT_DIR, 'Table3b_postdischarge_change_by_inhospital_category.csv'), index=False)
stack_scopes('evolution').to_csv(os.path.join(OUT_DIR, 'Table5_inhospital_aneurysm_status_by_window.csv'), index=False)
tS3 = pd.concat([stack_scopes('illday', 'illness_day').assign(population='Cohort C1'),
                 stack_scopes('illday_uncomplicated', 'illness_day').assign(population='Uncomplicated in hospital (Z<2)')], ignore_index=True)
tS3.to_csv(os.path.join(OUT_DIR, 'TableS3_status_by_illness_day.csv'), index=False)
tS4 = pd.concat([RES[s]['inhosp_wording_x_cat'].reindex(index=['normal', 'dilation', 'aneurysm'], columns=CATS, fill_value=0).rename_axis(index='inhospital_report_wording', columns=None)
                 .reset_index().assign(scope=s) for s in ['primary', 'all']], ignore_index=True)
tS4[['scope', 'inhospital_report_wording'] + CATS].to_csv(os.path.join(OUT_DIR, 'TableS4_inhospital_wording_vs_AHA.csv'), index=False)
print(t3a.to_string(index=False)); print(t3b.to_string(index=False))

IDX = {'Figure1_surveillance_ladder_and_first_documentation': 'A, yield of sequential echoes after an uncomplicated course (involvement and aneurysm); B, window in which involvement / aneurysm / medium+ was first documented.',
       'Figure2_risk_of_aneurysm_at_first_postdischarge_echo': 'A, stratified rates of aneurysm at the first <=30-d echo in children discharged with Z<2.5; B, Firth multivariable odds ratios.',
       'Figure3_course_of_inhospital_findings_by_size': 'A, aneurysm still present (Z>=2.5) by in-hospital size; B, lumen not yet normal (Z>=2); C, progression to a higher category.',
       'Figure4_new_aneurysm_after_discharge': 'Cumulative incidence of a new aneurysm after an uncomplicated course by risk-factor count (A) and after in-hospital dilation only vs Z<2 (B).',
       'Figure5_status_by_illness_day': 'Involvement and aneurysm by illness-day window (A, cohort C1; B, uncomplicated in hospital).',
       'Figure6_illness_day_of_first_documentation': 'Cumulative distribution of the illness day of the first report of involvement / aneurysm.',
       'FigureS1_cohort_flow': 'Cohort flow (primary scope).', 'FigureS2_planB_primary_vs_alldata': 'Plan B: primary (numeric era) vs all-data estimates.',
       'Table1 / TableS1': 'Characteristics by outcome; selection (echo <=30 d vs not).',
       'Table2_sequential_yield.csv': 'Sequential yields after an uncomplicated in-hospital course (R2).',
       'Table3a / Table3b': 'Window in which involvement / aneurysm / medium+ was first documented; post-discharge change by in-hospital category (R1).',
       'Table4a / Table4b': 'Stratified yields; Firth models.',
       'Table5_inhospital_aneurysm_status_by_window.csv': 'Status of in-hospital aneurysms by size and follow-up window (descriptive companion to Figure 3).',
       'Table6_time_to_event.csv': 'All Kaplan–Meier estimates (both scopes).', 'Table7_planB_primary_vs_alldata.csv': 'Plan B comparison.', 'TableS0': 'Numeric coverage by month (defines the era).',
       'TableS2_sensitivity.csv': 'Sensitivity analyses of the first-echo yield.',
       'TableS3_status_by_illness_day.csv': 'Involvement / aneurysm by illness-day window (R6).', 'TableS4_inhospital_wording_vs_AHA.csv': 'In-hospital report wording vs AHA category.', 'Results_locked_v4.txt / key_numbers.json': 'All key numbers.'}
open(os.path.join(OUT_DIR, 'Figures_and_tables_index_v4.txt'), 'w', encoding='utf-8').write('\n'.join(f'{k}: {v}' for k, v in IDX.items()))
print('outputs:', sorted(f for f in os.listdir(OUT_DIR) if f != 'cache'))